# Detección Automática de Clusters Coherentes

Analiza los **401 clusters** generados y detecta automáticamente cuáles son coherentes (productos similares).

## Objetivos:
1. **Calcular coherencia**: Medir homogeneidad de cada cluster usando keywords frecuentes
2. **Sugerir nombres**: Usar LLM para proponer nombres de categorías específicas
3. **Generar categorías**: Crear SQLs INSERT para poblar `gold_categorias`
4. **Asignar productos**: Insertar en `gold_productos_categorias` por cluster_id

## Salidas:
- `gold_cluster_analysis`: Análisis de coherencia de 401 clusters
- `gold_categorias_nuevas_sql`: Scripts SQL para crear categorías
- Estimado: **140-200 categorías ultra-específicas** (papel_higienico, yogur, papas_fritas, etc)

In [0]:
from pyspark.sql.functions import col, count, collect_list, struct, lit, current_timestamp
import pandas as pd
import numpy as np
from collections import Counter
import re
import requests
import json

print("✅ Librerías importadas")

In [0]:
# Cargar productos con sus clusters
productos_clusters = spark.sql("""
    SELECT 
        pc.cluster_id,
        p.id_producto,
        p.producto
    FROM workspace.superapp.gold_productos_clusters pc
    JOIN workspace.superapp.silver_prices p
        ON pc.id_producto = p.id_producto
""").toPandas()

print(f"📊 Productos cargados: {len(productos_clusters):,}")
print(f"📊 Número de clusters: {productos_clusters['cluster_id'].nunique()}")

# Calcular tamaño de clusters
cluster_sizes = productos_clusters.groupby('cluster_id').size().reset_index(name='size')
print(f"\n📊 Distribución de tamaños:")
print(f"   Min: {cluster_sizes['size'].min()}")
print(f"   Promedio: {cluster_sizes['size'].mean():.1f}")
print(f"   Mediana: {cluster_sizes['size'].median():.0f}")
print(f"   Max: {cluster_sizes['size'].max()}")

In [0]:
def calculate_cluster_coherence(cluster_products):
    """
    Calcula score de coherencia basado en keywords frecuentes.
    Retorna: (coherence_score, top_keywords, is_coherent)
    
    coherence_score: 0-1 (1 = muy coherente)
    top_keywords: lista de (keyword, count)
    is_coherent: True si score > threshold
    """
    # Normalizar textos
    texts = [p.lower() for p in cluster_products]
    all_text = ' '.join(texts)
    
    # Extraer palabras significativas (> 3 caracteres, sin números)
    words = re.findall(r'\b[a-záéíóúüñ]{4,}\b', all_text)
    
    if len(words) == 0:
        return 0.0, [], False
    
    # Contar frecuencias
    word_counts = Counter(words)
    
    # Excluir stopwords comunes
    stopwords = {'para', 'con', 'sin', 'pack', 'pack', 'grms', 'litro', 'botella', 'caja', 
                 'bolsa', 'pote', 'frasco', 'lata', 'sobre', 'unidad', 'unidades',
                 'carrefour', 'marca', 'precio', 'gran', 'compra', 'clasico', 'classic',
                 'sabor', 'saborizada', 'natural', 'original'}
    
    filtered_counts = {k: v for k, v in word_counts.items() if k not in stopwords}
    
    if len(filtered_counts) == 0:
        return 0.0, [], False
    
    # Top 5 keywords
    top_keywords = sorted(filtered_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    
    # Calcular score de coherencia:
    # - Si las top 3 keywords aparecen en > 50% de productos = coherente
    # - Si keyword dominante aparece en > 70% = muy coherente
    
    n_products = len(cluster_products)
    top_word, top_count = top_keywords[0] if top_keywords else ('', 0)
    
    # % de productos que contienen la palabra más frecuente
    products_with_top = sum(1 for p in texts if top_word in p)
    coverage = products_with_top / n_products if n_products > 0 else 0
    
    # Score: proporción de productos cubiertos por keyword dominante
    coherence_score = coverage
    
    # Threshold: coherente si > 60% de productos comparten keyword dominante
    is_coherent = coherence_score >= 0.60
    
    return coherence_score, top_keywords, is_coherent

print("✅ Función de coherencia definida")

In [0]:
def suggest_category_name_with_llm(cluster_products, top_keywords):
    """
    Usa LLM para sugerir nombre de categoría específico.
    Retorna: nombre sugerido (snake_case)
    """
    try:
        # Obtener token y workspace URL
        token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
        
        # Preparar samples (primeros 15 productos)
        samples = cluster_products[:15]
        samples_text = "\n".join([f"- {s}" for s in samples])
        
        # Keywords más frecuentes
        keywords_text = ", ".join([f"{kw}({cnt})" for kw, cnt in top_keywords[:3]])
        
        prompt = f"""Analiza estos productos de supermercado argentino y sugiere un nombre de CATEGORÍA ESPECÍFICO.

Productos:
{samples_text}

Keywords frecuentes: {keywords_text}

Ejemplos de buenos nombres:
- papel_higienico (no "limpieza")
- yogur_bebible (no "lacteos")
- papas_fritas (no "snacks")
- acondicionador_cabello (no "cuidado_personal")
- agua_mineral (no "bebidas")

Responde SOLO con el nombre de categoría en formato snake_case (2-4 palabras máximo).
Si los productos son muy diversos, responde "mixto"."""

        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
        
        data = {
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 30,
            "temperature": 0.2
        }
        
        response = requests.post(
            f"https://{workspace_url}/serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations",
            headers=headers,
            json=data,
            timeout=30
        )
        
        if response.status_code == 200:
            result = response.json()
            suggested = result['choices'][0]['message']['content'].strip().lower()
            # Limpiar respuesta
            suggested = re.sub(r'[^a-z_]', '', suggested)
            return suggested if suggested else "mixto"
        else:
            return "mixto"
            
    except Exception as e:
        return "mixto"

print("✅ Función de sugerencia LLM definida")

In [0]:
# Cargar productos sin clasificar
productos_sin_cat = spark.sql("""
    SELECT DISTINCT p.id_producto, p.producto
    FROM workspace.superapp.silver_prices p
    LEFT JOIN workspace.superapp.gold_productos_categorias pc 
        ON p.id_producto = pc.id_producto
    WHERE pc.id_categoria IS NULL
""").toPandas()

print(f"📊 Total productos sin clasificar: {len(productos_sin_cat):,}")
print(f"\nMuestra:")
for i, p in enumerate(productos_sin_cat['producto'].head(5), 1):
    print(f"  {i}. {p}")

In [0]:
import re
from collections import Counter

def extract_ngrams(text, n=2):
    """Extrae n-gramas de un texto."""
    words = re.findall(r'\b[a-záéíóúüñ]{3,}\b', text.lower())
    return [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]

def clean_ngram(ngram):
    """Filtra stopwords y palabras no significativas."""
    stopwords = {
        # Conectores y preposiciones
        'con', 'sin', 'para', 'por', 'del', 'los', 'las', 'una', 'uno', 'que',
        # Envases y medidas
        'pack', 'caja', 'bolsa', 'pote', 'frasco', 'lata', 'sobre', 'botella',
        'gramos', 'grs', 'litros', 'lts', 'unidades', 'uni', 'kilos',
        'grm', 'grs', 'mlt', 'bot', 'lat', 'sob', 'paq', 'doy', 'fco',
        # Términos genéricos
        'marca', 'precio', 'gran', 'compra', 'essential', 'selection',
        'clasico', 'classic', 'original', 'sabor', 'saborizada', 'natural',
        'premium', 'super', 'extra', 'light', 'diet', 'zero',
        # Supermercados argentinos
        'carrefour', 'coto', 'jumbo', 'disco', 'walmart', 'libertad',
        # Marcas alimenticias argentinas/internacionales
        'arcor', 'nestle', 'bimbo', 'molinos', 'marolio', 'glutano',
        'la', 'serenisima', 'sancor', 'ilolay', 'tregar', 'milkaut',
        'coca', 'cola', 'pepsi', 'fanta', 'sprite', 'quilmes', 'brahma',
        'dos', 'anclas', 'alicante', 'natura', 'knorr', 'maggi',
        # Marcas de cuidado personal/limpieza
        'colgate', 'dove', 'pantene', 'sedal', 'head', 'shoulders',
        'johnson', 'nivea', 'rexona', 'axe', 'skip', 'ala', 'magistral',
        'ayudin', 'procenex', 'poett', 'lysoform', 'glade', 'blem',
        # Marcas de ropa/hogar (Carrefour TEX)
        'tex', 'home', 'basic', 'detroit', 'arco', 'iris',
        # Marcas específicas detectadas
        'bon', 'key', 'kolor', 'zed', 'black', 'sensodyne', 'oral',
        'lays', 'krachitos', 'nikitos', 'pringles',
        # Marcas detectadas en análisis previo
        'silver', 'shadow', 'siempre', 'libre', 'moto', 'saint', 'gottard',
        # Términos de producto muy genéricos
        'producto', 'articulo', 'item'
    }
    
    words = ngram.split()
    # Filtrar si alguna palabra es stopword
    if any(w in stopwords for w in words):
        return None
    # Filtrar si todas las palabras tienen < 3 caracteres
    if all(len(w) < 3 for w in words):
        return None
    return ngram

print("✅ Funciones de n-gramas definidas (filtro marcas v2)")

In [0]:
print("\n🔍 Analizando bigramas en 73K productos...\n")

# Extraer todos los bigramas
all_bigrams = []
for producto in productos_sin_cat['producto']:
    bigrams = extract_ngrams(producto, n=2)
    all_bigrams.extend(bigrams)

print(f"   Total bigramas extraídos: {len(all_bigrams):,}")

# Limpiar y contar
cleaned_bigrams = [clean_ngram(bg) for bg in all_bigrams]
cleaned_bigrams = [bg for bg in cleaned_bigrams if bg is not None]

print(f"   Bigramas válidos (post-filtrado): {len(cleaned_bigrams):,}")

# Contar frecuencias
bigram_counts = Counter(cleaned_bigrams)

print(f"   Bigramas únicos: {len(bigram_counts):,}")
print(f"\n✅ Análisis completado")

In [0]:
print("\n" + "="*100)
print("🎯 TOP 10 BIGRAMAS MÁS FRECUENTES (Candidatos a Categoría)")
print("="*100)

top_bigrams = bigram_counts.most_common(20)

for rank, (bigram, count) in enumerate(top_bigrams[:10], 1):
    print(f"\n{'='*100}")
    print(f"{rank}. '{bigram.upper()}' → {count:,} productos ({count/len(productos_sin_cat)*100:.1f}%)")
    print("="*100)
    
    # Mostrar 10 productos ejemplo que contienen este bigrama
    ejemplos = productos_sin_cat[
        productos_sin_cat['producto'].str.lower().str.contains(bigram, regex=False)
    ]['producto'].head(10)
    
    print("\n🔍 Ejemplos de productos:")
    for i, ejemplo in enumerate(ejemplos, 1):
        print(f"   {i:2d}. {ejemplo}")
    
    print(f"\n📝 Categoría sugerida: {bigram.replace(' ', '_')}")

print("\n" + "="*100)

In [0]:
print("\n" + "="*100)
print("🎯 TOP 10 TRIGRAMAS MÁS FRECUENTES (Candidatos a Categoría)")
print("="*100)

# Extraer trigramas
print("\n🔍 Analizando trigramas...")
all_trigrams = []
for producto in productos_sin_cat['producto']:
    trigrams = extract_ngrams(producto, n=3)
    all_trigrams.extend(trigrams)

cleaned_trigrams = [clean_ngram(tg) for tg in all_trigrams]
cleaned_trigrams = [tg for tg in cleaned_trigrams if tg is not None]

trigram_counts = Counter(cleaned_trigrams)
top_trigrams = trigram_counts.most_common(20)

print(f"   Total trigramas únicos: {len(trigram_counts):,}\n")

for rank, (trigram, count) in enumerate(top_trigrams[:10], 1):
    print(f"\n{'='*100}")
    print(f"{rank}. '{trigram.upper()}' → {count:,} productos ({count/len(productos_sin_cat)*100:.1f}%)")
    print("="*100)
    
    # Mostrar 8 productos ejemplo
    ejemplos = productos_sin_cat[
        productos_sin_cat['producto'].str.lower().str.contains(trigram, regex=False)
    ]['producto'].head(8)
    
    print("\n🔍 Ejemplos de productos:")
    for i, ejemplo in enumerate(ejemplos, 1):
        print(f"   {i:2d}. {ejemplo}")
    
    print(f"\n📝 Categoría sugerida: {trigram.replace(' ', '_')}")

print("\n" + "="*100)

In [0]:
# Categorías válidas del top 50 (excluyendo marcas manualmente)
categorias_validas = [
    # Top 1-25
    ('crema_dental', 'Crema Dental', 'Pastas dentales y dentífricos'),
    ('papas_fritas', 'Papas Fritas', 'Snacks de papas fritas en paquete'),
    ('cepillo_dental', 'Cepillo Dental', 'Cepillos de dientes manuales'),
    ('limpiador_liq', 'Limpiador Líquido', 'Limpiadores líquidos multiuso para pisos'),
    ('color_perm', 'Tintura Permanente', 'Tinturas para cabello permanentes'),
    ('jab_liq', 'Jabón Líquido', 'Jabones líquidos para tocador y ropa'),
    ('alimento_perros', 'Alimento Perros', 'Alimento seco para perros'),
    ('jugo_polvo', 'Jugo en Polvo', 'Jugos instantáneos en polvo'),
    ('alimento_gatos', 'Alimento Gatos', 'Alimento seco para gatos'),
    ('celular_libre', 'Celular Libre', 'Teléfonos celulares liberados'),
    ('jabon_tocador', 'Jabón Tocador', 'Jabones en barra para tocador'),
    ('limpiador_liquido', 'Limpiador Líquido', 'Limpiadores líquidos concentrados'),
    ('jabon_liq_2', 'Jabón Líquido Ropa', 'Jabones líquidos para lavado de ropa'),
    ('agua_gas', 'Agua Gasificada', 'Agua con gas saborizada'),
    ('enjuague_bucal', 'Enjuague Bucal', 'Enjuagues bucales antisépticos'),
    ('mate_cocido', 'Mate Cocido', 'Mate cocido en saquitos'),
    ('aceitunas_verdes', 'Aceitunas Verdes', 'Aceitunas verdes enteras o rellenas'),
    ('sal_fina', 'Sal Fina', 'Sal fina de mesa'),
    ('dulce_leche', 'Dulce de Leche', 'Dulce de leche y productos con DDL'),
    ('agua_mineral', 'Agua Mineral', 'Agua mineral con y sin gas'),
    ('crema_corporal', 'Crema Corporal', 'Cremas hidratantes corporales'),
    ('jabon_liquido_2', 'Jabón Líquido', 'Jabones líquidos varios usos'),
    ('rollo_cocina', 'Rollo Cocina', 'Rollos de papel de cocina'),
    ('dulce_batata', 'Dulce de Batata', 'Dulce de batata en estuche o lata'),
    ('papel_higienico', 'Papel Higiénico', 'Papel higiénico simple y doble hoja'),
    ('trapo_piso', 'Trapo de Piso', 'Trapos para limpieza de pisos'),
    # Top 26-50
    ('toalla_fem', 'Toalla Femenina', 'Toallas higiénicas femeninas'),
    ('tableta_chocolate', 'Tableta Chocolate', 'Tabletas y barras de chocolate'),
    ('pan_rallado', 'Pan Rallado', 'Pan rallado común y sin gluten'),
    ('huevo_chocolate', 'Huevo Chocolate', 'Huevos de chocolate con sorpresa'),
    ('plato_playo', 'Plato Playo', 'Platos playos de cerámica y porcelana'),
    ('crema_facial', 'Crema Facial', 'Cremas faciales hidratantes y antiarrugas'),
    ('barra_cereal', 'Barra Cereal', 'Barras de cereal y granola'),
    ('tapa_empanada', 'Tapa Empanada', 'Tapas para empanadas fritas y al horno'),
    ('crema_peinar', 'Crema Peinar', 'Cremas para peinar y definir'),
    ('huevo_pascua', 'Huevo Pascua', 'Huevos de pascua de chocolate'),
    ('leche_polvo', 'Leche en Polvo', 'Leche en polvo infantil y adultos'),
    ('chocolate_blanco', 'Chocolate Blanco', 'Chocolates y alfajores de chocolate blanco'),
    ('toallas_fem_2', 'Toallas Femeninas', 'Toallas higiénicas femeninas'),
    ('aloe_vera', 'Aloe Vera', 'Productos con aloe vera'),
    ('maquina_afeitar', 'Máquina Afeitar', 'Máquinas de afeitar descartables'),
    ('toallitas_humedas', 'Toallitas Húmedas', 'Toallitas húmedas para bebés'),
    ('snack_perros', 'Snack Perros', 'Golosinas y snacks para perros'),
    ('alfajor_chocolate', 'Alfajor Chocolate', 'Alfajores bañados en chocolate'),
    ('agua_sab', 'Agua Saborizada', 'Aguas saborizadas con y sin gas'),
    ('lampara_led', 'Lámpara LED', 'Lámparas LED varias potencias'),
    ('carpeta_escolar', 'Carpeta Escolar', 'Carpetas escolares 3 anillos'),
    ('plato_postre', 'Plato Postre', 'Platos de postre cerámica y vidrio'),
]

print(f"\n🔧 GENERANDO SQLs PARA {len(categorias_validas)} CATEGORÍAS\n")
print("="*100)

# Generar INSERTs
for i, (id_cat, nombre, descripcion) in enumerate(categorias_validas, start=26):
    sql = f"""
INSERT INTO workspace.superapp.gold_categorias 
    (id_categoria, nombre, nivel, parent_id, descripcion, fecha_creacion)
VALUES 
    ({i}, '{nombre}', 'categoria', NULL, '{descripcion}', current_timestamp());
"""
    print(sql)

print("="*100)
print(f"\n✅ {len(categorias_validas)} SQLs generados. ID desde 26 hasta {25 + len(categorias_validas)}")
print(f"\n📝 Próximo paso: Ejecutar estos INSERTs en gold_categorias")

In [0]:
# Mapeo bigrama -> id_subcategoria para QUESO (parent_id=18)
subcategorias_queso = [
    # Prioridad alta (más específicos primero)
    ('queso crema', 74),
    ('queso untable', 74),  # mismo que crema
    ('queso unt', 74),  # variante
    ('queso rallado', 76),
    ('queso azul', 78),
    ('queso fundido', 79),
    ('queso mozzarella', 81),
    ('queso hebras', 82),
    ('port salut', 80),
    ('queso port', 80),
    ('queso pategras', 83),
    ('pategras', 83),
    ('queso sardo', 84),
    ('queso reggianito', 85),
    ('reggianito', 85),
    ('queso cheddar', 86),
    ('cheddar', 86),
    ('mozzarella', 81),
    ('sardo', 84),
    ('queso cremoso', 77),  # genérico al final
]

print("\n🧀 REASIGNANDO PRODUCTOS DE QUESO A SUBCATEGORÍAS")
print("="*100)

# Crear tabla temporal con todas las asignaciones
from pyspark.sql.functions import lit, row_number
from pyspark.sql.window import Window

asignaciones_queso = []
for bigram, id_subcat in subcategorias_queso:
    query = f"""
        SELECT 
            p.id_producto,
            {id_subcat} as id_subcategoria,
            '{bigram}' as bigram_match
        FROM workspace.superapp.silver_prices p
        JOIN workspace.superapp.gold_productos_categorias pc 
            ON p.id_producto = pc.id_producto
        WHERE pc.id_categoria = 18
        AND LOWER(p.producto) LIKE '%{bigram}%'
    """
    df_match = spark.sql(query)
    asignaciones_queso.append(df_match)

# Union de todas las asignaciones
if asignaciones_queso:
    df_all = asignaciones_queso[0]
    for df in asignaciones_queso[1:]:
        df_all = df_all.union(df)
    
    # Eliminar duplicados: si un producto matchea múltiples bigramas, usar el primero (más específico)
    window_spec = Window.partitionBy('id_producto').orderBy('bigram_match')
    df_unique = df_all.withColumn('rn', row_number().over(window_spec)) \
                      .filter('rn = 1') \
                      .select('id_producto', 'id_subcategoria')
    
    df_unique.createOrReplaceTempView('temp_queso_subcats')
    
    # MERGE con source sin duplicados
    spark.sql("""
        MERGE INTO workspace.superapp.gold_productos_categorias AS target
        USING temp_queso_subcats AS source
        ON target.id_producto = source.id_producto
        WHEN MATCHED THEN
            UPDATE SET id_subcategoria = source.id_subcategoria
    """)
    
    total = df_unique.count()
    print(f"\n✅ {total:,} productos de queso reasignados a subcategorías")
    
    # Mostrar distribución
    result = spark.sql("""
        SELECT 
            c.nombre as subcategoria,
            COUNT(*) as total
        FROM workspace.superapp.gold_productos_categorias pc
        JOIN workspace.superapp.gold_categorias c ON pc.id_subcategoria = c.id_categoria
        WHERE pc.id_categoria = 18 AND pc.id_subcategoria IS NOT NULL
        GROUP BY c.nombre
        ORDER BY total DESC
    """)
    
    print("\n📊 Distribución por subcategoría de queso:\n")
    display(result)
else:
    print("\n⚠️ No se encontraron productos de queso para reasignar")

print("="*100)

In [0]:
# Mapeo bigrama -> id_subcategoria para CAFÉ (parent_id=10)
subcategorias_cafe = [
    # Prioridad alta (más específicos primero)
    ('cafe capsula', 89),
    ('cafe capsulas', 89),
    ('nespresso', 89),
    ('dolce gusto', 89),
    ('cafe grano', 90),
    ('cafe granos', 90),
    ('cafe tostado grano', 90),
    ('cafe torrado', 91),
    ('cafe instantaneo', 88),
    ('cafe soluble', 88),
    ('cafe molido', 87),  # genérico al final
]

print("\n☕ REASIGNANDO PRODUCTOS DE CAFÉ A SUBCATEGORÍAS")
print("="*100)

# Crear tabla temporal con todas las asignaciones
from pyspark.sql.functions import lit, row_number
from pyspark.sql.window import Window

asignaciones_cafe = []
for bigram, id_subcat in subcategorias_cafe:
    query = f"""
        SELECT 
            p.id_producto,
            {id_subcat} as id_subcategoria,
            '{bigram}' as bigram_match
        FROM workspace.superapp.silver_prices p
        JOIN workspace.superapp.gold_productos_categorias pc 
            ON p.id_producto = pc.id_producto
        WHERE pc.id_categoria = 10
        AND LOWER(p.producto) LIKE '%{bigram}%'
    """
    df_match = spark.sql(query)
    asignaciones_cafe.append(df_match)

# Union de todas las asignaciones
if asignaciones_cafe:
    df_all = asignaciones_cafe[0]
    for df in asignaciones_cafe[1:]:
        df_all = df_all.union(df)
    
    # Eliminar duplicados: si un producto matchea múltiples bigramas, usar el primero (más específico)
    window_spec = Window.partitionBy('id_producto').orderBy('bigram_match')
    df_unique = df_all.withColumn('rn', row_number().over(window_spec)) \
                      .filter('rn = 1') \
                      .select('id_producto', 'id_subcategoria')
    
    df_unique.createOrReplaceTempView('temp_cafe_subcats')
    
    # MERGE con source sin duplicados
    spark.sql("""
        MERGE INTO workspace.superapp.gold_productos_categorias AS target
        USING temp_cafe_subcats AS source
        ON target.id_producto = source.id_producto
        WHEN MATCHED THEN
            UPDATE SET id_subcategoria = source.id_subcategoria
    """)
    
    total = df_unique.count()
    print(f"\n✅ {total:,} productos de café reasignados a subcategorías")
    
    # Mostrar distribución
    result = spark.sql("""
        SELECT 
            c.nombre as subcategoria,
            COUNT(*) as total
        FROM workspace.superapp.gold_productos_categorias pc
        JOIN workspace.superapp.gold_categorias c ON pc.id_subcategoria = c.id_categoria
        WHERE pc.id_categoria = 10 AND pc.id_subcategoria IS NOT NULL
        GROUP BY c.nombre
        ORDER BY total DESC
    """)
    
    print("\n📊 Distribución por subcategoría de café:\n")
    display(result)
else:
    print("\n⚠️ No se encontraron productos de café para reasignar")

print("="*100)

In [0]:
%sql
-- RESUMEN FINAL DE RECATEGORIZACIÓN

SELECT 
    c.id_categoria,
    c.nombre,
    c.nivel,
    COUNT(pc.id_producto) as total_productos,
    ROUND(COUNT(pc.id_producto) * 100.0 / (SELECT COUNT(*) FROM workspace.superapp.gold_productos_categorias WHERE id_categoria IS NOT NULL), 2) as porcentaje
FROM workspace.superapp.gold_categorias c
LEFT JOIN workspace.superapp.gold_productos_categorias pc 
    ON c.id_categoria = pc.id_categoria
WHERE c.id_categoria >= 92  -- Sólo las nuevas categorías
   OR c.id_categoria IN (44, 62, 36, 31)  -- Y las categorías existentes que recibieron productos
GROUP BY c.id_categoria, c.nombre, c.nivel
ORDER BY total_productos DESC;

In [0]:
print("\n🔧 CORRIGIENDO PRODUCTOS MAL CLASIFICADOS EN LECHE")
print("="*100)

# 1. Mover dulce de leche a categoría 44 (ya existe)
print("\n1️⃣ Moviendo dulce de leche a categoría 44...")

# Obtener lista de IDs a actualizar
ids_dulce = spark.sql("""
    SELECT DISTINCT pc.id_producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.silver_prices p ON pc.id_producto = p.id_producto
    WHERE pc.id_categoria = 4
    AND (LOWER(p.producto) LIKE '%dulce%leche%' OR LOWER(p.producto) LIKE '%dulce de leche%')
""").collect()

if ids_dulce:
    ids_list = [f"'{row.id_producto}'" for row in ids_dulce]
    ids_str = ", ".join(ids_list)
    
    spark.sql(f"""
        UPDATE workspace.superapp.gold_productos_categorias
        SET id_categoria = 44
        WHERE id_producto IN ({ids_str})
    """)
    print(f"   ✅ {len(ids_dulce):,} productos de dulce de leche movidos a categoría 44")
else:
    print("   ⚠️  No se encontraron productos de dulce de leche")

# 2. Mover leche en polvo a categoría 62 (ya existe como subcategoría)
print("\n2️⃣ Moviendo leche en polvo a categoría 62...")

ids_polvo = spark.sql("""
    SELECT DISTINCT pc.id_producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.silver_prices p ON pc.id_producto = p.id_producto
    WHERE pc.id_categoria = 4
    AND LOWER(p.producto) LIKE '%leche%polvo%'
""").collect()

if ids_polvo:
    ids_list = [f"'{row.id_producto}'" for row in ids_polvo]
    ids_str = ", ".join(ids_list)
    
    spark.sql(f"""
        UPDATE workspace.superapp.gold_productos_categorias
        SET id_categoria = 62
        WHERE id_producto IN ({ids_str})
    """)
    print(f"   ✅ {len(ids_polvo):,} productos de leche en polvo movidos a categoría 62")
else:
    print("   ⚠️  No se encontraron productos de leche en polvo")

# 3. Quitar chocolate/tabletas de LECHE (poner id_categoria = NULL)
print("\n3️⃣ Quitando chocolate/tabletas de categoría LECHE...")

ids_chocolate = spark.sql("""
    SELECT DISTINCT pc.id_producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.silver_prices p ON pc.id_producto = p.id_producto
    WHERE pc.id_categoria = 4
    AND (LOWER(p.producto) LIKE '%chocolate%' OR LOWER(p.producto) LIKE '%tableta%')
""").collect()

if ids_chocolate:
    ids_list = [f"'{row.id_producto}'" for row in ids_chocolate]
    ids_str = ", ".join(ids_list)
    
    spark.sql(f"""
        UPDATE workspace.superapp.gold_productos_categorias
        SET id_categoria = NULL
        WHERE id_producto IN ({ids_str})
    """)
    print(f"   ✅ {len(ids_chocolate):,} productos de chocolate removidos de LECHE")
else:
    print("   ⚠️  No se encontraron productos de chocolate en LECHE")

# Verificar qué queda en LECHE (id=4)
remaining = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria = 4
""").collect()[0]['cnt']

print(f"\n📊 Productos restantes en LECHE (id=4): {remaining:,}")
print("   Estos deberían ser leches genéricas sin especificar tipo\n")

# Verificar distribuciones finales
print("\n📊 Verificación Final:")

dist = spark.sql("""
    SELECT 
        c.nombre,
        COUNT(*) as total
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.gold_categorias c ON pc.id_categoria = c.id_categoria
    WHERE pc.id_categoria IN (4, 44, 62)
    GROUP BY c.nombre
    ORDER BY total DESC
""")

display(dist)

print("="*100)

In [0]:
%sql
-- RESUMEN GLOBAL FINAL: Estado completo del proyecto de categorización

-- 1. Total de categorías creadas
SELECT '✅ Total Categorías Creadas' as metrica, COUNT(*) as valor
FROM workspace.superapp.gold_categorias
WHERE nivel = 'categoria'

UNION ALL

-- 2. Total de subcategorías creadas
SELECT '✅ Total Subcategorías Creadas' as metrica, COUNT(*) as valor
FROM workspace.superapp.gold_categorias
WHERE nivel = 'subcategoria'

UNION ALL

-- 3. Productos clasificados
SELECT '✅ Productos Clasificados' as metrica, COUNT(*) as valor
FROM workspace.superapp.gold_productos_categorias
WHERE id_categoria IS NOT NULL

UNION ALL

-- 4. Productos sin clasificar
SELECT '⚠️  Productos Sin Clasificar' as metrica, COUNT(*) as valor
FROM workspace.superapp.gold_productos_categorias
WHERE id_categoria IS NULL

UNION ALL

-- 5. Total productos
SELECT '📊 Total Productos' as metrica, COUNT(*) as valor
FROM workspace.superapp.gold_productos_categorias;

-- TOP 20 Categorías con más productos
SELECT 
    c.id_categoria,
    c.nombre,
    COUNT(pc.id_producto) as total_productos,
    ROUND(COUNT(pc.id_producto) * 100.0 / (
        SELECT COUNT(*) 
        FROM workspace.superapp.gold_productos_categorias 
        WHERE id_categoria IS NOT NULL
    ), 2) as porcentaje
FROM workspace.superapp.gold_categorias c
LEFT JOIN workspace.superapp.gold_productos_categorias pc 
    ON c.id_categoria = pc.id_categoria
WHERE c.nivel = 'categoria'
GROUP BY c.id_categoria, c.nombre
ORDER BY total_productos DESC
LIMIT 20;

# 🚀 Próximos Pasos: Iteraciones y Optimización

## Iteración 2: Label Propagation Recursivo
**Idea**: Usar los nuevos productos clasificados como semillas adicionales
- Volver a ejecutar celdas 1-5 
- Los productos recién asignados ahora son parte de "clasificados"
- Threshold puede ser más permisivo (0.70) en segunda iteración
- **Objetivo**: +10-15K productos adicionales

## Estrategia 3: Extraer Más Bigramas
**Para los que no matchean con Label Propagation:**
- Extraer bigramas de productos aún sin clasificar
- Crear 30-50 categorías ultra-específicas adicionales
- Aplicar keyword matching
- **Objetivo**: +5-8K productos

## Estrategia 4: Clasificación Manual Asistida por LLM
**Para categorías importantes con pocos productos:**
- Seleccionar top 100 productos sin clasificar más frecuentes
- Usar LLM para sugerir categoría en batch
- Revisar manualmente y aplicar
- **Objetivo**: +2-3K productos de alta frecuencia

## Meta Final
🎯 **60-70% de cobertura** (53-62K productos clasificados) en 2-3 iteraciones

---

## Ajustes de Threshold

| Threshold | Precisión | Cobertura | Uso Recomendado |
|-----------|-----------|-----------|------------------|
| 0.85      | Muy Alta  | Baja      | Iteración 1 (conservador) |
| 0.75      | Alta      | Media     | **Iteración 1 (balanceado)** |
| 0.70      | Media     | Alta      | Iteración 2+ (más permisivo) |
| 0.60      | Baja      | Muy Alta  | Solo con revisión manual |

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

print("\n" + "="*100)
print("🎯 LABEL PROPAGATION - CLASIFICACIÓN SEMI-SUPERVISADA")
print("="*100)

# ============================================================================
# 1. CARGAR DATOS (ÚNICOS POR NOMBRE DE PRODUCTO)
# ============================================================================
print("\n1️⃣ Cargando productos únicos...")

# Clasificados: tomar categoría más frecuente por producto
df_clasificados_raw = spark.sql("""
    SELECT DISTINCT p.producto, pc.id_categoria
    FROM workspace.superapp.silver_prices p
    JOIN workspace.superapp.gold_productos_categorias pc ON p.id_producto = pc.id_producto
    WHERE pc.id_categoria IS NOT NULL
""").toPandas()

df_clasificados = df_clasificados_raw.groupby('producto')['id_categoria'].agg(
    lambda x: x.value_counts().index[0]
).reset_index()

# Sin clasificar
df_sin_clasificar = spark.sql("""
    SELECT DISTINCT p.producto
    FROM workspace.superapp.silver_prices p
    LEFT JOIN workspace.superapp.gold_productos_categorias pc ON p.id_producto = pc.id_producto
    WHERE pc.id_categoria IS NULL
""").toPandas()

print(f"   ✅ Clasificados: {len(df_clasificados):,} productos únicos")
print(f"   ⚠️  Sin clasificar: {len(df_sin_clasificar):,} productos únicos")

# ============================================================================
# 2. VECTORIZAR CON TF-IDF
# ============================================================================
print("\n2️⃣ Vectorizando con TF-IDF (char n-grams)...")

vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=5000,
    lowercase=True,
    strip_accents='unicode'
)

X_clasificados = vectorizer.fit_transform(df_clasificados['producto'])
X_sin_clasificar = vectorizer.transform(df_sin_clasificar['producto'])

print(f"   ✅ Clasificados: {X_clasificados.shape}")
print(f"   ✅ Sin clasificar: {X_sin_clasificar.shape}")

# ============================================================================
# 3. COMPUTAR SIMILITUD Y ASIGNAR (BATCH PROCESSING)
# ============================================================================
print("\n3️⃣ Computando similitud coseno y asignando categorías...")

SIMILARITY_THRESHOLD = 0.75
BATCH_SIZE = 5000

print(f"   Parámetros: Threshold={SIMILARITY_THRESHOLD}, Batch={BATCH_SIZE:,}\n")

asignaciones = []
n_batches = (len(df_sin_clasificar) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in range(n_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(df_sin_clasificar))
    
    X_batch = X_sin_clasificar[start_idx:end_idx]
    similarities = cosine_similarity(X_batch, X_clasificados)
    
    for i in range(len(X_batch)):
        max_sim_idx = similarities[i].argmax()
        max_similarity = similarities[i][max_sim_idx]
        
        if max_similarity >= SIMILARITY_THRESHOLD:
            producto_actual = df_sin_clasificar.iloc[start_idx + i]['producto']
            categoria_asignada = df_clasificados.iloc[max_sim_idx]['id_categoria']
            producto_similar = df_clasificados.iloc[max_sim_idx]['producto']
            
            asignaciones.append({
                'producto': producto_actual,
                'id_categoria': categoria_asignada,
                'similitud': max_similarity,
                'producto_ref': producto_similar
            })
    
    if (batch_idx + 1) % 2 == 0 or batch_idx == n_batches - 1:
        print(f"   Batch {batch_idx + 1}/{n_batches} | Asignados: {len(asignaciones):,}")

print(f"\n   ✅ Total asignaciones: {len(asignaciones):,} ({len(asignaciones)/len(df_sin_clasificar)*100:.1f}%)")

# ============================================================================
# 4. ANÁLISIS DE ASIGNACIONES
# ============================================================================
if len(asignaciones) > 0:
    df_asignaciones = pd.DataFrame(asignaciones)
    
    print("\n4️⃣ Análisis de asignaciones:\n")
    print(f"   Similitud mínima:  {df_asignaciones['similitud'].min():.4f}")
    print(f"   Similitud promedio: {df_asignaciones['similitud'].mean():.4f}")
    print(f"   Similitud máxima:   {df_asignaciones['similitud'].max():.4f}")
    
    print("\n   Top 10 categorías con más asignaciones:\n")
    cat_names_df = spark.sql("SELECT id_categoria, nombre FROM workspace.superapp.gold_categorias").toPandas()
    
    for cat_id, count in df_asignaciones['id_categoria'].value_counts().head(10).items():
        cat_name = cat_names_df[cat_names_df['id_categoria'] == cat_id]['nombre'].values[0]
        print(f"   {cat_id:3d}. {cat_name:<30} → {count:>5,} productos")
    
    print("\n   Ejemplos (alta similitud > 0.90):\n")
    for _, row in df_asignaciones[df_asignaciones['similitud'] > 0.90].head(5).iterrows():
        cat_name = cat_names_df[cat_names_df['id_categoria'] == row['id_categoria']]['nombre'].values[0]
        print(f"   {row['similitud']:.3f} | {cat_name}")
        print(f"      Nuevo: {row['producto'][:70]}")
        print(f"      Ref:   {row['producto_ref'][:70]}\n")
    
    # ============================================================================
    # 5. APLICAR ASIGNACIONES A TODOS LOS id_producto
    # ============================================================================
    print("\n5️⃣ Aplicando asignaciones a base de datos...\n")
    
    total_actualizados = 0
    
    for id_cat in df_asignaciones['id_categoria'].unique():
        productos_cat = df_asignaciones[df_asignaciones['id_categoria'] == id_cat]['producto'].tolist()
        
        # Escapar comillas simples en nombres de productos
        productos_escaped = [p.replace("'", "''") for p in productos_cat]
        productos_str = "', '".join(productos_escaped)
        
        # UPDATE: asignar categoría a TODOS los id_producto con estos nombres
        spark.sql(f"""
            UPDATE workspace.superapp.gold_productos_categorias pc
            SET 
                id_categoria = {id_cat},
                metodo = 'label_propagation',
                confianza = 0.80,
                fecha_asignacion = current_timestamp(),
                notas = 'Asignado por TF-IDF similitud'
            WHERE pc.id_producto IN (
                SELECT id_producto
                FROM workspace.superapp.silver_prices
                WHERE producto IN ('{productos_str}')
            )
            AND pc.id_categoria IS NULL
        """)
        
        total_actualizados += len(productos_cat)
        print(f"   ✅ Cat {id_cat:3d}: {len(productos_cat):>4} productos únicos actualizados")
    
    print(f"\n   🎉 Total: {total_actualizados:,} productos únicos reasignados")
    
    # ============================================================================
    # 6. ESTADO FINAL
    # ============================================================================
    print("\n" + "="*100)
    print("📊 ESTADO FINAL DE CLASIFICACIÓN")
    print("="*100 + "\n")
    
    estado_final = spark.sql("""
        SELECT 
            CASE WHEN id_categoria IS NOT NULL THEN 'Clasificados' ELSE 'Sin Clasificar' END as estado,
            COUNT(DISTINCT id_producto) as productos_unicos,
            ROUND(COUNT(DISTINCT id_producto) * 100.0 / SUM(COUNT(DISTINCT id_producto)) OVER(), 2) as porcentaje
        FROM workspace.superapp.gold_productos_categorias
        GROUP BY CASE WHEN id_categoria IS NOT NULL THEN 'Clasificados' ELSE 'Sin Clasificar' END
        ORDER BY estado DESC
    """)
    
    display(estado_final)
    
else:
    print("\n⚠️  No se encontraron asignaciones con similitud > threshold")

print("\n" + "="*100)

In [0]:
print("\n💾 APLICANDO ASIGNACIONES A BASE DE DATOS")
print("="*100)

if len(asignaciones) > 0:
    df_asignaciones = pd.DataFrame(asignaciones)
    
    print(f"\n⚠️  IMPORTANTE: Se van a actualizar {len(df_asignaciones):,} productos")
    print("\nPresiona Enter para continuar o Ctrl+C para cancelar...")
    # input()  # Descomentar para confirmación manual
    
    print("\n🔄 Procesando asignaciones por categoría...\n")
    
    # Agrupar por categoría para hacer UPDATEs eficientes
    total_actualizados = 0
    
    for id_cat in df_asignaciones['id_categoria'].unique():
        productos_cat = df_asignaciones[df_asignaciones['id_categoria'] == id_cat]
        
        # Crear lista de IDs para UPDATE
        ids_list = [f"'{pid}'" for pid in productos_cat['id_producto']]
        ids_str = ", ".join(ids_list)
        
        # UPDATE con IN clause
        spark.sql(f"""
            UPDATE workspace.superapp.gold_productos_categorias
            SET 
                id_categoria = {id_cat},
                metodo = 'label_propagation',
                confianza = 0.85,
                fecha_asignacion = current_timestamp(),
                notas = 'Asignado por similitud TF-IDF'
            WHERE id_producto IN ({ids_str})
        """)
        
        total_actualizados += len(productos_cat)
        print(f"   ✅ Categoría {id_cat:3d}: {len(productos_cat):>5,} productos actualizados")
    
    print(f"\n✅ Total productos actualizados: {total_actualizados:,}")
    
    # Verificar estado final
    print("\n" + "="*100)
    print("📊 ESTADO FINAL DE CLASIFICACIÓN")
    print("="*100)
    
    estado_final = spark.sql("""
        SELECT 
            CASE 
                WHEN id_categoria IS NOT NULL THEN 'Clasificados'
                ELSE 'Sin Clasificar'
            END as estado,
            COUNT(*) as total_productos,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
        FROM workspace.superapp.gold_productos_categorias
        GROUP BY CASE WHEN id_categoria IS NOT NULL THEN 'Clasificados' ELSE 'Sin Clasificar' END
        ORDER BY estado DESC
    """).toPandas()
    
    print("\n")
    display(estado_final)
    
    clasificados = estado_final[estado_final['estado'] == 'Clasificados']['total_productos'].values[0]
    porcentaje = estado_final[estado_final['estado'] == 'Clasificados']['porcentaje'].values[0]
    
    print(f"\n🎉 NUEVO TOTAL CLASIFICADOS: {clasificados:,} productos ({porcentaje}%)")
    print(f"🚀 Ganancia en esta fase: +{len(asignaciones):,} productos")
    
else:
    print("\n⚠️  No hay asignaciones para aplicar")

print("\n" + "="*100)

In [0]:
print("\n🔍 ANALIZANDO ASIGNACIONES PROPUESTAS")
print("="*100)

if len(asignaciones) > 0:
    df_asignaciones = pd.DataFrame(asignaciones)
    
    # 1. Estadísticas de similitud
    print("\n1️⃣ Estadísticas de Similitud:")
    print(f"   - Mínima: {df_asignaciones['similitud'].min():.4f}")
    print(f"   - Promedio: {df_asignaciones['similitud'].mean():.4f}")
    print(f"   - Mediana: {df_asignaciones['similitud'].median():.4f}")
    print(f"   - Máxima: {df_asignaciones['similitud'].max():.4f}")
    
    # 2. Distribución por categoría
    print("\n2️⃣ Top 15 Categorías con más asignaciones nuevas:\n")
    
    cat_counts = df_asignaciones['id_categoria'].value_counts().head(15)
    
    # Obtener nombres de categorías
    cat_names = spark.sql("""
        SELECT id_categoria, nombre
        FROM workspace.superapp.gold_categorias
    """).toPandas()
    
    for cat_id, count in cat_counts.items():
        cat_name = cat_names[cat_names['id_categoria'] == cat_id]['nombre'].values[0]
        porcentaje = count / len(asignaciones) * 100
        print(f"   {cat_id:3d}. {cat_name:<30} → {count:>5,} productos ({porcentaje:>5.1f}%)")
    
    # 3. Ejemplos de asignaciones (alta similitud)
    print("\n3️⃣ Ejemplos de asignaciones con ALTA similitud (> 0.90):\n")
    
    ejemplos_alta = df_asignaciones[df_asignaciones['similitud'] > 0.90].head(10)
    
    for i, row in ejemplos_alta.iterrows():
        cat_name = cat_names[cat_names['id_categoria'] == row['id_categoria']]['nombre'].values[0]
        print(f"   Similitud: {row['similitud']:.3f} | Cat: {cat_name}")
        print(f"   → Nuevo:      {row['producto']}")
        print(f"   → Referencia: {row['producto_referencia']}")
        print()
    
    # 4. Ejemplos de asignaciones (baja similitud - límite threshold)
    print("\n4️⃣ Ejemplos de asignaciones con BAJA similitud (0.75-0.80):\n")
    
    ejemplos_baja = df_asignaciones[
        (df_asignaciones['similitud'] >= 0.75) & 
        (df_asignaciones['similitud'] <= 0.80)
    ].head(10)
    
    for i, row in ejemplos_baja.iterrows():
        cat_name = cat_names[cat_names['id_categoria'] == row['id_categoria']]['nombre'].values[0]
        print(f"   Similitud: {row['similitud']:.3f} | Cat: {cat_name}")
        print(f"   → Nuevo:      {row['producto']}")
        print(f"   → Referencia: {row['producto_referencia']}")
        print()
    
else:
    print("\n⚠️  No se encontraron asignaciones con similitud > threshold")

print("="*100)

In [0]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("\n🎯 COMPUTANDO SIMILITUD COSENO Y ASIGNANDO CATEGORÍAS")
print("="*100)

# Parámetros
SIMILARITY_THRESHOLD = 0.75  # Solo asignar si similitud > 75%
BATCH_SIZE = 5000            # Procesar 5K productos a la vez

print(f"\n🛠️  Parámetros:")
print(f"   - Threshold de similitud: {SIMILARITY_THRESHOLD}")
print(f"   - Batch size: {BATCH_SIZE:,}")

# Resultado: lista de asignaciones
asignaciones = []

# Procesar en batches
n_batches = (len(df_sin_clasificar) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\n🔄 Procesando {len(df_sin_clasificar):,} productos en {n_batches} batches...\n")

for batch_idx in range(n_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(df_sin_clasificar))
    
    # Batch actual
    X_batch = X_sin_clasificar[start_idx:end_idx]
    
    # Computar similitud: batch x clasificados
    similarities = cosine_similarity(X_batch, X_clasificados)
    
    # Para cada producto en el batch, encontrar el más similar
    for i, (idx, row) in enumerate(df_sin_clasificar.iloc[start_idx:end_idx].iterrows()):
        # Índice del producto clasificado más similar
        max_sim_idx = similarities[i].argmax()
        max_similarity = similarities[i][max_sim_idx]
        
        # Solo asignar si similitud > threshold
        if max_similarity >= SIMILARITY_THRESHOLD:
            # Obtener categoría del producto más similar
            categoria_asignada = df_clasificados.iloc[max_sim_idx]['id_categoria']
            producto_similar = df_clasificados.iloc[max_sim_idx]['producto']
            
            asignaciones.append({
                'id_producto': row['id_producto'],
                'producto': row['producto'],
                'id_categoria': categoria_asignada,
                'similitud': max_similarity,
                'producto_referencia': producto_similar
            })
    
    # Progress
    if (batch_idx + 1) % 3 == 0 or batch_idx == n_batches - 1:
        print(f"   Batch {batch_idx + 1}/{n_batches} completado | Asignaciones hasta ahora: {len(asignaciones):,}")

print(f"\n✅ Procesamiento completado!")
print(f"\n📊 Resultados:")
print(f"   - Productos analizados: {len(df_sin_clasificar):,}")
print(f"   - Productos asignados (sim > {SIMILARITY_THRESHOLD}): {len(asignaciones):,}")
print(f"   - Cobertura adicional: {len(asignaciones)/len(df_sin_clasificar)*100:.2f}%")

print("="*100)

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

print("\n🧮 VECTORIZANDO PRODUCTOS CON TF-IDF")
print("="*100)

# Configurar TF-IDF
# - char_ngrams (2,4): captura variaciones de escritura ("yogur"/"yogurt")
# - word_ngrams (1,2): captura bigramas importantes ("dulce leche")
# - max_features: limitar dimensionalidad para eficiencia

vectorizer = TfidfVectorizer(
    analyzer='char_wb',  # char n-grams con word boundaries
    ngram_range=(2, 4),  # bigramas y trigramas de caracteres
    max_features=5000,   # top 5000 features
    lowercase=True,
    strip_accents='unicode'
)

print("\n1️⃣ Entrenando TF-IDF con productos clasificados...")

# Fit TF-IDF solo con productos clasificados
X_clasificados = vectorizer.fit_transform(df_clasificados['producto'])

print(f"   ✅ Matriz clasificados: {X_clasificados.shape}")
print(f"   ✅ Features extraidos: {len(vectorizer.get_feature_names_out())}")

# Transform productos sin clasificar
print("\n2️⃣ Transformando productos sin clasificar...")
X_sin_clasificar = vectorizer.transform(df_sin_clasificar['producto'])

print(f"   ✅ Matriz sin clasificar: {X_sin_clasificar.shape}")

print("\n✅ Vectorización completada!")
print("="*100)

In [0]:
print("\n📊 PREPARANDO DATOS PARA LABEL PROPAGATION")
print("="*100)

# Cargar productos CLASIFICADOS (semillas)
df_clasificados = spark.sql("""
    SELECT 
        p.id_producto,
        p.producto,
        pc.id_categoria
    FROM workspace.superapp.silver_prices p
    JOIN workspace.superapp.gold_productos_categorias pc
        ON p.id_producto = pc.id_producto
    WHERE pc.id_categoria IS NOT NULL
""").toPandas()

# Cargar productos SIN CLASIFICAR
df_sin_clasificar = spark.sql("""
    SELECT DISTINCT
        p.id_producto,
        p.producto
    FROM workspace.superapp.silver_prices p
    LEFT JOIN workspace.superapp.gold_productos_categorias pc
        ON p.id_producto = pc.id_producto
    WHERE pc.id_categoria IS NULL
""").toPandas()

print(f"\n✅ Productos CLASIFICADOS (semillas): {len(df_clasificados):,}")
print(f"⚠️  Productos SIN CLASIFICAR:          {len(df_sin_clasificar):,}")
print(f"\n📊 Distribución de categorías semilla:")
print(df_clasificados['id_categoria'].value_counts().head(10))

print("\n" + "="*100)

# Fase 5: Label Propagation - Clasificación Semi-Supervisada

## Estrategia
Usar los **28,252 productos ya clasificados** como "semillas" para clasificar los **60,412 restantes** mediante similitud de texto.

## Enfoque Híbrido
1. **TF-IDF Vectorization**: Más efectivo que embeddings semánticos para nombres de productos retail
2. **Similitud Coseno**: Comparar cada producto sin clasificar vs productos clasificados
3. **Asignación con Threshold**: Solo asignar si similitud > 0.75 (alta confianza)
4. **Iterativo**: Repetir proceso usando nuevas asignaciones como semillas

## Ventajas
- ✅ No requiere entrenar modelos complejos
- ✅ Aprovecha las 28K clasificaciones existentes
- ✅ TF-IDF captura mejor variaciones de productos retail (marcas, tamaños)
- ✅ Threshold ajustable según precisión deseada

## Objetivo
Clasificar **20-30K productos adicionales** (total: ~50K clasificados = 56%+)

In [0]:
print("\n🔧 CORRIGIENDO PRODUCTOS MAL CLASIFICADOS EN LECHE")
print("="*100)

# 1. Mover dulce de leche a categoría 44 (ya existe)
print("\n1️⃣ Moviendo dulce de leche a categoría 44...")
spark.sql("""
    UPDATE workspace.superapp.gold_productos_categorias
    SET id_categoria = 44
    WHERE id_categoria = 4
    AND id_producto IN (
        SELECT p.id_producto
        FROM workspace.superapp.silver_prices p
        WHERE LOWER(p.producto) LIKE '%dulce%leche%'
           OR LOWER(p.producto) LIKE '%dulce de leche%'
    )
""")

count_dulce = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria = 44
""").collect()[0]['cnt']

print(f"   ✅ Categoría 44 (Dulce de Leche) ahora tiene {count_dulce:,} productos")

# 2. Mover leche en polvo a categoría 62 (ya existe como subcategoría)
print("\n2️⃣ Moviendo leche en polvo a categoría 62...")
spark.sql("""
    UPDATE workspace.superapp.gold_productos_categorias
    SET id_categoria = 62
    WHERE id_categoria = 4
    AND id_producto IN (
        SELECT p.id_producto
        FROM workspace.superapp.silver_prices p
        WHERE LOWER(p.producto) LIKE '%leche%polvo%'
    )
""")

count_polvo = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria = 62
""").collect()[0]['cnt']

print(f"   ✅ Categoría 62 (Leche en Polvo) ahora tiene {count_polvo:,} productos")

# 3. Quitar chocolate/tabletas de LECHE (poner id_categoria = NULL)
print("\n3️⃣ Quitando chocolate/tabletas de categoría LECHE...")
spark.sql("""
    UPDATE workspace.superapp.gold_productos_categorias
    SET id_categoria = NULL
    WHERE id_categoria = 4
    AND id_producto IN (
        SELECT p.id_producto
        FROM workspace.superapp.silver_prices p
        WHERE (LOWER(p.producto) LIKE '%chocolate%' 
           OR LOWER(p.producto) LIKE '%tableta%')
        AND id_producto IN (
            SELECT id_producto 
            FROM workspace.superapp.gold_productos_categorias 
            WHERE id_categoria = 4
        )
    )
""")

print(f"   ✅ Productos de chocolate removidos de LECHE")

# Verificar qué queda en LECHE (id=4)
remaining = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria = 4
""").collect()[0]['cnt']

print(f"\n📊 Productos restantes en LECHE (id=4): {remaining:,}")
print("   Estos deberían ser leches genéricas sin especificar tipo\n")

print("="*100)

In [0]:
# Mapeo bigrama -> nueva id_categoria para FIDEOS (original id=3)
categorias_fideos = [
    ('spaghetti', 103),
    ('penne', 104),
    ('mostachol', 104),
    ('tirabuzon', 105),
    ('fusilli', 105),
]

print("\n🍝 REASIGNANDO FIDEOS A CATEGORÍAS ESPECÍFICAS")
print("="*100)

from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

asignaciones_fideos = []
for bigram, nueva_cat in categorias_fideos:
    query = f"""
        SELECT 
            p.id_producto,
            {nueva_cat} as nueva_categoria,
            '{bigram}' as bigram_match
        FROM workspace.superapp.silver_prices p
        JOIN workspace.superapp.gold_productos_categorias pc 
            ON p.id_producto = pc.id_producto
        WHERE pc.id_categoria = 3
        AND LOWER(p.producto) LIKE '%{bigram}%'
    """
    df_match = spark.sql(query)
    asignaciones_fideos.append(df_match)

if asignaciones_fideos:
    df_all = asignaciones_fideos[0]
    for df in asignaciones_fideos[1:]:
        df_all = df_all.union(df)
    
    window_spec = Window.partitionBy('id_producto').orderBy('bigram_match')
    df_unique = df_all.withColumn('rn', row_number().over(window_spec)) \
                      .filter('rn = 1') \
                      .select('id_producto', 'nueva_categoria')
    
    df_unique.createOrReplaceTempView('temp_fideos_cats')
    
    spark.sql("""
        MERGE INTO workspace.superapp.gold_productos_categorias AS target
        USING temp_fideos_cats AS source
        ON target.id_producto = source.id_producto
        WHEN MATCHED THEN
            UPDATE SET id_categoria = source.nueva_categoria
    """)
    
    total = df_unique.count()
    print(f"\n✅ {total:,} productos de fideos reasignados")
    
    result = spark.sql("""
        SELECT 
            c.nombre as categoria,
            COUNT(*) as total
        FROM workspace.superapp.gold_productos_categorias pc
        JOIN workspace.superapp.gold_categorias c ON pc.id_categoria = c.id_categoria
        WHERE pc.id_categoria IN (103, 104, 105)
        GROUP BY c.nombre
        ORDER BY total DESC
    """)
    
    print("\n📊 Distribución fideos por categoría:\n")
    display(result)

print("="*100)

In [0]:
# Mapeo bigrama -> nueva id_categoria para JABÓN (original id=7)
categorias_jabon = [
    ('jabon polvo', 102),  # Nueva categoría
    ('jabon tocador', 36),  # Ya existe
    ('jabon liq', 31),      # Ya existe  
    ('jabon liquido', 31),  # Ya existe
]

print("\n🧴 REASIGNANDO JABÓN A CATEGORÍAS ESPECÍFICAS")
print("="*100)

from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

asignaciones_jabon = []
for bigram, nueva_cat in categorias_jabon:
    query = f"""
        SELECT 
            p.id_producto,
            {nueva_cat} as nueva_categoria,
            '{bigram}' as bigram_match
        FROM workspace.superapp.silver_prices p
        JOIN workspace.superapp.gold_productos_categorias pc 
            ON p.id_producto = pc.id_producto
        WHERE pc.id_categoria = 7
        AND LOWER(p.producto) LIKE '%{bigram}%'
    """
    df_match = spark.sql(query)
    asignaciones_jabon.append(df_match)

if asignaciones_jabon:
    df_all = asignaciones_jabon[0]
    for df in asignaciones_jabon[1:]:
        df_all = df_all.union(df)
    
    window_spec = Window.partitionBy('id_producto').orderBy('bigram_match')
    df_unique = df_all.withColumn('rn', row_number().over(window_spec)) \
                      .filter('rn = 1') \
                      .select('id_producto', 'nueva_categoria')
    
    df_unique.createOrReplaceTempView('temp_jabon_cats')
    
    spark.sql("""
        MERGE INTO workspace.superapp.gold_productos_categorias AS target
        USING temp_jabon_cats AS source
        ON target.id_producto = source.id_producto
        WHEN MATCHED THEN
            UPDATE SET id_categoria = source.nueva_categoria
    """)
    
    total = df_unique.count()
    print(f"\n✅ {total:,} productos de jabón reasignados")
    
    result = spark.sql("""
        SELECT 
            c.nombre as categoria,
            COUNT(*) as total
        FROM workspace.superapp.gold_productos_categorias pc
        JOIN workspace.superapp.gold_categorias c ON pc.id_categoria = c.id_categoria
        WHERE pc.id_categoria IN (102, 36, 31)
        GROUP BY c.nombre
        ORDER BY total DESC
    """)
    
    print("\n📊 Distribución jabón por categoría:\n")
    display(result)

print("="*100)

In [0]:
# Mapeo bigrama -> nueva id_categoria para LECHE (original id=4)
categorias_leche = [
    ('leche chocolatada', 99),
    ('leche entera', 100),
    ('crema leche', 101),
    ('crema de leche', 101),
]

print("\n🥛 REASIGNANDO LECHE A CATEGORÍAS ESPECÍFICAS")
print("="*100)

from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

asignaciones_leche = []
for bigram, nueva_cat in categorias_leche:
    query = f"""
        SELECT 
            p.id_producto,
            {nueva_cat} as nueva_categoria,
            '{bigram}' as bigram_match
        FROM workspace.superapp.silver_prices p
        JOIN workspace.superapp.gold_productos_categorias pc 
            ON p.id_producto = pc.id_producto
        WHERE pc.id_categoria = 4
        AND LOWER(p.producto) LIKE '%{bigram}%'
        AND LOWER(p.producto) NOT LIKE '%dulce%'  -- Excluir dulce de leche
        AND LOWER(p.producto) NOT LIKE '%chocolate%'  -- Excluir chocolate tabletas
    """
    df_match = spark.sql(query)
    asignaciones_leche.append(df_match)

if asignaciones_leche:
    df_all = asignaciones_leche[0]
    for df in asignaciones_leche[1:]:
        df_all = df_all.union(df)
    
    window_spec = Window.partitionBy('id_producto').orderBy('bigram_match')
    df_unique = df_all.withColumn('rn', row_number().over(window_spec)) \
                      .filter('rn = 1') \
                      .select('id_producto', 'nueva_categoria')
    
    df_unique.createOrReplaceTempView('temp_leche_cats')
    
    spark.sql("""
        MERGE INTO workspace.superapp.gold_productos_categorias AS target
        USING temp_leche_cats AS source
        ON target.id_producto = source.id_producto
        WHEN MATCHED THEN
            UPDATE SET id_categoria = source.nueva_categoria
    """)
    
    total = df_unique.count()
    print(f"\n✅ {total:,} productos de leche válidos reasignados")
    
    result = spark.sql("""
        SELECT 
            c.nombre as categoria,
            COUNT(*) as total
        FROM workspace.superapp.gold_productos_categorias pc
        JOIN workspace.superapp.gold_categorias c ON pc.id_categoria = c.id_categoria
        WHERE pc.id_categoria IN (99, 100, 101)
        GROUP BY c.nombre
        ORDER BY total DESC
    """)
    
    print("\n📊 Distribución leche por categoría:\n")
    display(result)

print("\n⚠️  Productos mal clasificados en LECHE serán reasignados después")
print("="*100)

In [0]:
# Mapeo bigrama -> nueva id_categoria para GALLETITAS (original id=8)
categorias_galletitas = [
    ('galletitas dulces', 95),
    ('galletitas crackers', 96),
    ('galletitas chocolate', 97),
    ('galletitas salvado', 98),
]

print("\n🍪 REASIGNANDO GALLETITAS A CATEGORÍAS ESPECÍFICAS")
print("="*100)

from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

asignaciones_gallet = []
for bigram, nueva_cat in categorias_galletitas:
    query = f"""
        SELECT 
            p.id_producto,
            {nueva_cat} as nueva_categoria,
            '{bigram}' as bigram_match
        FROM workspace.superapp.silver_prices p
        JOIN workspace.superapp.gold_productos_categorias pc 
            ON p.id_producto = pc.id_producto
        WHERE pc.id_categoria = 8
        AND LOWER(p.producto) LIKE '%{bigram}%'
    """
    df_match = spark.sql(query)
    asignaciones_gallet.append(df_match)

if asignaciones_gallet:
    df_all = asignaciones_gallet[0]
    for df in asignaciones_gallet[1:]:
        df_all = df_all.union(df)
    
    window_spec = Window.partitionBy('id_producto').orderBy('bigram_match')
    df_unique = df_all.withColumn('rn', row_number().over(window_spec)) \
                      .filter('rn = 1') \
                      .select('id_producto', 'nueva_categoria')
    
    df_unique.createOrReplaceTempView('temp_gallet_cats')
    
    spark.sql("""
        MERGE INTO workspace.superapp.gold_productos_categorias AS target
        USING temp_gallet_cats AS source
        ON target.id_producto = source.id_producto
        WHEN MATCHED THEN
            UPDATE SET id_categoria = source.nueva_categoria
    """)
    
    total = df_unique.count()
    print(f"\n✅ {total:,} productos de galletitas reasignados")
    
    result = spark.sql("""
        SELECT 
            c.nombre as categoria,
            COUNT(*) as total
        FROM workspace.superapp.gold_productos_categorias pc
        JOIN workspace.superapp.gold_categorias c ON pc.id_categoria = c.id_categoria
        WHERE pc.id_categoria IN (95, 96, 97, 98)
        GROUP BY c.nombre
        ORDER BY total DESC
    """)
    
    print("\n📊 Distribución galletitas por categoría:\n")
    display(result)

print("="*100)

In [0]:
# Mapeo bigrama -> nueva id_categoria para VINO (original id=16)
categorias_vino = [
    ('vino tinto', 92),
    ('tinto malbec', 92),
    ('tinto cabernet', 92),
    ('vino malbec', 92),
    ('cabernet sauvignon', 92),
    ('vino blanco', 93),
    ('blanco dulce', 93),
    ('vino rosado', 94),
]

print("\n🍷 REASIGNANDO VINOS A CATEGORÍAS ESPECÍFICAS")
print("="*100)

from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

asignaciones_vino = []
for bigram, nueva_cat in categorias_vino:
    query = f"""
        SELECT 
            p.id_producto,
            {nueva_cat} as nueva_categoria,
            '{bigram}' as bigram_match
        FROM workspace.superapp.silver_prices p
        JOIN workspace.superapp.gold_productos_categorias pc 
            ON p.id_producto = pc.id_producto
        WHERE pc.id_categoria = 16
        AND LOWER(p.producto) LIKE '%{bigram}%'
    """
    df_match = spark.sql(query)
    asignaciones_vino.append(df_match)

if asignaciones_vino:
    df_all = asignaciones_vino[0]
    for df in asignaciones_vino[1:]:
        df_all = df_all.union(df)
    
    # Eliminar duplicados: priorizar primera coincidencia
    window_spec = Window.partitionBy('id_producto').orderBy('bigram_match')
    df_unique = df_all.withColumn('rn', row_number().over(window_spec)) \
                      .filter('rn = 1') \
                      .select('id_producto', 'nueva_categoria')
    
    df_unique.createOrReplaceTempView('temp_vino_cats')
    
    # UPDATE categoría
    spark.sql("""
        MERGE INTO workspace.superapp.gold_productos_categorias AS target
        USING temp_vino_cats AS source
        ON target.id_producto = source.id_producto
        WHEN MATCHED THEN
            UPDATE SET id_categoria = source.nueva_categoria
    """)
    
    total = df_unique.count()
    print(f"\n✅ {total:,} productos de vino reasignados")
    
    # Mostrar distribución
    result = spark.sql("""
        SELECT 
            c.nombre as categoria,
            COUNT(*) as total
        FROM workspace.superapp.gold_productos_categorias pc
        JOIN workspace.superapp.gold_categorias c ON pc.id_categoria = c.id_categoria
        WHERE pc.id_categoria IN (92, 93, 94)
        GROUP BY c.nombre
        ORDER BY total DESC
    """)
    
    print("\n📊 Distribución vinos por categoría:\n")
    display(result)

print("="*100)

In [0]:
%sql
-- Crear 14 categorías específicas para subdividir las grandes (IDs 92-105)

INSERT INTO workspace.superapp.gold_categorias (id_categoria, nombre, nivel, parent_id, descripcion, fecha_creacion) VALUES 
-- VINO (3 categorías)
(92, 'Vino Tinto', 'categoria', NULL, 'Vinos tintos (malbec, cabernet, etc)', current_timestamp()),
(93, 'Vino Blanco', 'categoria', NULL, 'Vinos blancos (semillon, chardonnay, etc)', current_timestamp()),
(94, 'Vino Rosado', 'categoria', NULL, 'Vinos rosados y rosé', current_timestamp()),

-- GALLETITAS (4 categorías)
(95, 'Galletitas Dulces', 'categoria', NULL, 'Galletitas y galletas dulces', current_timestamp()),
(96, 'Galletitas Crackers', 'categoria', NULL, 'Galletitas crackers y saladas', current_timestamp()),
(97, 'Galletitas Chocolate', 'categoria', NULL, 'Galletitas con chocolate o rellenas', current_timestamp()),
(98, 'Galletitas Salvado', 'categoria', NULL, 'Galletitas de salvado integrales', current_timestamp()),

-- LECHE (3 categorías)
(99, 'Leche Chocolatada', 'categoria', NULL, 'Leche saborizada con chocolate', current_timestamp()),
(100, 'Leche Entera', 'categoria', NULL, 'Leche entera líquida', current_timestamp()),
(101, 'Crema de Leche', 'categoria', NULL, 'Crema de leche para cocinar y batir', current_timestamp()),

-- JABÓN (1 categoría)
(102, 'Jabón en Polvo', 'categoria', NULL, 'Jabón en polvo para ropa', current_timestamp()),

-- FIDEOS (3 categorías)
(103, 'Fideos Spaghetti', 'categoria', NULL, 'Fideos tipo spaghetti', current_timestamp()),
(104, 'Fideos Penne', 'categoria', NULL, 'Fideos tipo penne y mostachol', current_timestamp()),
(105, 'Fideos Tirabuzon', 'categoria', NULL, 'Fideos tipo tirabuzon y fusilli', current_timestamp());

SELECT '✅ 14 categorías específicas creadas (IDs 92-105)' as resultado;

In [0]:
# Analizar las 5 categorías más grandes (excluyendo queso y café ya subdivididos)
categorias_grandes = [
    (16, 'vino', 4121),
    (4, 'leche', 1901),
    (8, 'galletitas', 1648),
    (7, 'jabon', 1602),
    (3, 'fideos', 1087),
]

print("\n" + "="*100)
print("🔍 ANÁLISIS DE CATEGORÍAS GRANDES PARA SUBDIVIDIR")
print("="*100)

for id_cat, nombre, count in categorias_grandes:
    print(f"\n\n{'#'*100}")
    print(f"## {nombre.upper()} - {count:,} productos (ID: {id_cat})")
    print("#"*100)
    
    # Cargar productos de esta categoría
    productos = spark.sql(f"""
        SELECT DISTINCT p.id_producto, p.producto
        FROM workspace.superapp.silver_prices p
        JOIN workspace.superapp.gold_productos_categorias pc 
            ON p.id_producto = pc.id_producto
        WHERE pc.id_categoria = {id_cat}
    """).toPandas()
    
    # Extraer bigramas
    all_bigrams = []
    for producto in productos['producto']:
        bigrams = extract_ngrams(producto, n=2)
        all_bigrams.extend(bigrams)
    
    cleaned_bigrams = [clean_ngram(bg) for bg in all_bigrams]
    cleaned_bigrams = [bg for bg in cleaned_bigrams if bg is not None]
    bigram_counts_cat = Counter(cleaned_bigrams)
    
    print(f"\n📄 TOP 10 SUBCATEGORÍAS DETECTADAS:\n")
    
    for rank, (bigram, bg_count) in enumerate(bigram_counts_cat.most_common(10), 1):
        porcentaje = bg_count / len(productos) * 100
        print(f"{rank:2d}. '{bigram.upper():<30}' → {bg_count:>4} productos ({porcentaje:>4.1f}%)")
        
        # Mostrar 3 ejemplos
        ejemplos = productos[
            productos['producto'].str.lower().str.contains(bigram, regex=False)
        ]['producto'].head(3)
        
        for ejemplo in ejemplos:
            print(f"     • {ejemplo}")
        print()

print("\n" + "="*100)
print("📝 RECOMENDACIONES DE SUBDIVISIÓN:")
print("="*100)
print("""
1. VINO (4,121 productos):
   ✅ vino_tinto, vino_blanco, vino_rosado, vino_espumante, champagne
   
2. LECHE (1,901 productos):
   ✅ leche_entera, leche_descremada, leche_deslactosada, leche_chocolatada
   
3. GALLETITAS (1,648 productos):
   ✅ galletitas_dulces, galletitas_saladas, galletitas_agua, galletitas_arroz
   
4. JABÓN (1,602 productos):
   ✅ jabon_polvo, jabon_liquido_ropa, jabon_tocador (ya existe como cat 36)
   
5. FIDEOS (1,087 productos):
   ✅ fideos_secos, fideos_frescos, fideos_rellenos
""")
print("="*100)

In [0]:
%sql
-- Insertar 18 subcategorías: 13 de queso + 5 de café

-- SUBCATEGORÍAS DE QUESO (parent_id = 18)
INSERT INTO workspace.superapp.gold_categorias (id_categoria, nombre, nivel, parent_id, descripcion, fecha_creacion) VALUES 
(74, 'Queso Crema', 'subcategoria', 18, 'Queso crema para untar', current_timestamp()),
(75, 'Queso Untable', 'subcategoria', 18, 'Quesos untables saborizados', current_timestamp()),
(76, 'Queso Rallado', 'subcategoria', 18, 'Quesos rallados duros (sardo, reggianito, parmesano)', current_timestamp()),
(77, 'Queso Cremoso', 'subcategoria', 18, 'Quesos cremosos de pasta blanda', current_timestamp()),
(78, 'Queso Azul', 'subcategoria', 18, 'Quesos azules (gorgonzola, roquefort)', current_timestamp()),
(79, 'Queso Fundido', 'subcategoria', 18, 'Quesos procesados fundidos en potes', current_timestamp()),
(80, 'Port Salut', 'subcategoria', 18, 'Queso Port Salut y variantes', current_timestamp()),
(81, 'Queso Mozzarella', 'subcategoria', 18, 'Queso mozzarella y pizza', current_timestamp()),
(82, 'Queso Hebras', 'subcategoria', 18, 'Quesos en hebras para gratinar', current_timestamp()),
(83, 'Queso Pategras', 'subcategoria', 18, 'Queso Pategras tipo argentino', current_timestamp()),
(84, 'Queso Sardo', 'subcategoria', 18, 'Queso Sardo en trozo o rallado', current_timestamp()),
(85, 'Queso Reggianito', 'subcategoria', 18, 'Queso Reggianito tipo parmesano', current_timestamp()),
(86, 'Queso Cheddar', 'subcategoria', 18, 'Queso Cheddar en fetas', current_timestamp()),

-- SUBCATEGORÍAS DE CAFÉ (parent_id = 10)
(87, 'Café Molido', 'subcategoria', 10, 'Café molido tostado', current_timestamp()),
(88, 'Café Instantáneo', 'subcategoria', 10, 'Café soluble instantáneo', current_timestamp()),
(89, 'Café Cápsulas', 'subcategoria', 10, 'Cápsulas de café (Nespresso, Dolce Gusto, etc)', current_timestamp()),
(90, 'Café Tostado Grano', 'subcategoria', 10, 'Café en grano tostado', current_timestamp()),
(91, 'Café Torrado', 'subcategoria', 10, 'Café molido torrado (con azúcar)', current_timestamp());

SELECT '✅ 18 subcategorías creadas correctamente' AS resultado;

In [0]:
%sql
-- Eliminar subcategoría duplicada "Queso Untable" (es lo mismo que Queso Crema)
DELETE FROM workspace.superapp.gold_categorias WHERE id_categoria = 75;

-- Actualizar descripción de Queso Crema para incluir untables
UPDATE workspace.superapp.gold_categorias 
SET 
    nombre = 'Queso Crema/Untable',
    descripcion = 'Queso crema y quesos untables saborizados'
WHERE id_categoria = 74;

SELECT '✅ Queso Crema y Untable consolidados en id=74' as resultado;

In [0]:
# Cargar productos de la categoría CAFE
productos_cafe = spark.sql("""
    SELECT DISTINCT p.id_producto, p.producto
    FROM workspace.superapp.silver_prices p
    JOIN workspace.superapp.gold_productos_categorias pc 
        ON p.id_producto = pc.id_producto
    WHERE pc.id_categoria = 10  -- cafe
""").toPandas()

print(f"\n☕ ANALIZANDO SUBCATEGORÍAS DE CAFÉ")
print(f"="*100)
print(f"Total productos café: {len(productos_cafe):,}\n")

# Extraer bigramas de productos de café
all_bigrams_cafe = []
for producto in productos_cafe['producto']:
    bigrams = extract_ngrams(producto, n=2)
    all_bigrams_cafe.extend(bigrams)

# Limpiar y contar
cleaned_bigrams_cafe = [clean_ngram(bg) for bg in all_bigrams_cafe]
cleaned_bigrams_cafe = [bg for bg in cleaned_bigrams_cafe if bg is not None]
bigram_counts_cafe = Counter(cleaned_bigrams_cafe)

print("\n📄 TOP 20 SUBCATEGORÍAS DETECTADAS:\n")

for rank, (bigram, count) in enumerate(bigram_counts_cafe.most_common(20), 1):
    porcentaje = count / len(productos_cafe) * 100
    print(f"{rank:2d}. '{bigram.upper():<25}' → {count:>4} productos ({porcentaje:>4.1f}%)")
    
    # Mostrar 3 ejemplos
    ejemplos = productos_cafe[
        productos_cafe['producto'].str.lower().str.contains(bigram, regex=False)
    ]['producto'].head(3)
    
    for ejemplo in ejemplos:
        print(f"     • {ejemplo}")
    print()

print("="*100)

In [0]:
%sql
-- Insertar 48 categorías detectadas por análisis de bigramas (IDs 26-73)

INSERT INTO workspace.superapp.gold_categorias (id_categoria, nombre, nivel, parent_id, descripcion, fecha_creacion) VALUES 
(26, 'Crema Dental', 'categoria', NULL, 'Pastas dentales y dentífricos', current_timestamp()),
(27, 'Papas Fritas', 'categoria', NULL, 'Snacks de papas fritas en paquete', current_timestamp()),
(28, 'Cepillo Dental', 'categoria', NULL, 'Cepillos de dientes manuales', current_timestamp()),
(29, 'Limpiador Líquido', 'categoria', NULL, 'Limpiadores líquidos multiuso para pisos', current_timestamp()),
(30, 'Tintura Permanente', 'categoria', NULL, 'Tinturas para cabello permanentes', current_timestamp()),
(31, 'Jabón Líquido', 'categoria', NULL, 'Jabones líquidos para tocador y ropa', current_timestamp()),
(32, 'Alimento Perros', 'categoria', NULL, 'Alimento seco para perros', current_timestamp()),
(33, 'Jugo en Polvo', 'categoria', NULL, 'Jugos instantáneos en polvo', current_timestamp()),
(34, 'Alimento Gatos', 'categoria', NULL, 'Alimento seco para gatos', current_timestamp()),
(35, 'Celular Libre', 'categoria', NULL, 'Teléfonos celulares liberados', current_timestamp()),
(36, 'Jabón Tocador', 'categoria', NULL, 'Jabones en barra para tocador', current_timestamp()),
(37, 'Limpiador Líquido Concentrado', 'categoria', NULL, 'Limpiadores líquidos concentrados', current_timestamp()),
(38, 'Jabón Líquido Ropa', 'categoria', NULL, 'Jabones líquidos para lavado de ropa', current_timestamp()),
(39, 'Agua Gasificada', 'categoria', NULL, 'Agua con gas saborizada', current_timestamp()),
(40, 'Enjuague Bucal', 'categoria', NULL, 'Enjuagues bucales antisépticos', current_timestamp()),
(41, 'Mate Cocido', 'categoria', NULL, 'Mate cocido en saquitos', current_timestamp()),
(42, 'Aceitunas Verdes', 'categoria', NULL, 'Aceitunas verdes enteras o rellenas', current_timestamp()),
(43, 'Sal Fina', 'categoria', NULL, 'Sal fina de mesa', current_timestamp()),
(44, 'Dulce de Leche', 'categoria', NULL, 'Dulce de leche y productos con DDL', current_timestamp()),
(45, 'Agua Mineral', 'categoria', NULL, 'Agua mineral con y sin gas', current_timestamp()),
(46, 'Crema Corporal', 'categoria', NULL, 'Cremas hidratantes corporales', current_timestamp()),
(47, 'Jabón Líquido Varios', 'categoria', NULL, 'Jabones líquidos varios usos', current_timestamp()),
(48, 'Rollo Cocina', 'categoria', NULL, 'Rollos de papel de cocina', current_timestamp()),
(49, 'Dulce de Batata', 'categoria', NULL, 'Dulce de batata en estuche o lata', current_timestamp()),
(50, 'Papel Higiénico', 'categoria', NULL, 'Papel higiénico simple y doble hoja', current_timestamp()),
(51, 'Trapo de Piso', 'categoria', NULL, 'Trapos para limpieza de pisos', current_timestamp()),
(52, 'Toalla Femenina', 'categoria', NULL, 'Toallas higiénicas femeninas', current_timestamp()),
(53, 'Tableta Chocolate', 'categoria', NULL, 'Tabletas y barras de chocolate', current_timestamp()),
(54, 'Pan Rallado', 'categoria', NULL, 'Pan rallado común y sin gluten', current_timestamp()),
(55, 'Huevo Chocolate', 'categoria', NULL, 'Huevos de chocolate con sorpresa', current_timestamp()),
(56, 'Plato Playo', 'categoria', NULL, 'Platos playos de cerámica y porcelana', current_timestamp()),
(57, 'Crema Facial', 'categoria', NULL, 'Cremas faciales hidratantes y antiarrugas', current_timestamp()),
(58, 'Barra Cereal', 'categoria', NULL, 'Barras de cereal y granola', current_timestamp()),
(59, 'Tapa Empanada', 'categoria', NULL, 'Tapas para empanadas fritas y al horno', current_timestamp()),
(60, 'Crema Peinar', 'categoria', NULL, 'Cremas para peinar y definir', current_timestamp()),
(61, 'Huevo Pascua', 'categoria', NULL, 'Huevos de pascua de chocolate', current_timestamp()),
(62, 'Leche en Polvo', 'categoria', NULL, 'Leche en polvo infantil y adultos', current_timestamp()),
(63, 'Chocolate Blanco', 'categoria', NULL, 'Chocolates y alfajores de chocolate blanco', current_timestamp()),
(64, 'Toallas Femeninas', 'categoria', NULL, 'Toallas higiénicas femeninas', current_timestamp()),
(65, 'Aloe Vera', 'categoria', NULL, 'Productos con aloe vera', current_timestamp()),
(66, 'Máquina Afeitar', 'categoria', NULL, 'Máquinas de afeitar descartables', current_timestamp()),
(67, 'Toallitas Húmedas', 'categoria', NULL, 'Toallitas húmedas para bebés', current_timestamp()),
(68, 'Snack Perros', 'categoria', NULL, 'Golosinas y snacks para perros', current_timestamp()),
(69, 'Alfajor Chocolate', 'categoria', NULL, 'Alfajores bañados en chocolate', current_timestamp()),
(70, 'Agua Saborizada', 'categoria', NULL, 'Aguas saborizadas con y sin gas', current_timestamp()),
(71, 'Lámpara LED', 'categoria', NULL, 'Lámparas LED varias potencias', current_timestamp()),
(72, 'Carpeta Escolar', 'categoria', NULL, 'Carpetas escolares 3 anillos', current_timestamp()),
(73, 'Plato Postre', 'categoria', NULL, 'Platos de postre cerámica y vidrio', current_timestamp());

SELECT '✅ 48 categorías insertadas correctamente' AS resultado;

In [0]:
# Cargar productos de la categoría QUESO
productos_queso = spark.sql("""
    SELECT DISTINCT p.id_producto, p.producto
    FROM workspace.superapp.silver_prices p
    JOIN workspace.superapp.gold_productos_categorias pc 
        ON p.id_producto = pc.id_producto
    WHERE pc.id_categoria = 18  -- queso
""").toPandas()

print(f"\n🧀 ANALIZANDO SUBCATEGORÍAS DE QUESO")
print(f"="*100)
print(f"Total productos queso: {len(productos_queso):,}\n")

# Extraer bigramas de productos de queso
all_bigrams_queso = []
for producto in productos_queso['producto']:
    bigrams = extract_ngrams(producto, n=2)
    all_bigrams_queso.extend(bigrams)

# Limpiar y contar
cleaned_bigrams_queso = [clean_ngram(bg) for bg in all_bigrams_queso]
cleaned_bigrams_queso = [bg for bg in cleaned_bigrams_queso if bg is not None]
bigram_counts_queso = Counter(cleaned_bigrams_queso)

print("\n📄 TOP 20 SUBCATEGORÍAS DETECTADAS:\n")

for rank, (bigram, count) in enumerate(bigram_counts_queso.most_common(20), 1):
    porcentaje = count / len(productos_queso) * 100
    print(f"{rank:2d}. '{bigram.upper():<25}' → {count:>4} productos ({porcentaje:>4.1f}%)")
    
    # Mostrar 3 ejemplos
    ejemplos = productos_queso[
        productos_queso['producto'].str.lower().str.contains(bigram, regex=False)
    ]['producto'].head(3)
    
    for ejemplo in ejemplos:
        print(f"     • {ejemplo}")
    print()

print("="*100)

In [0]:
# Mapeo bigrama -> id_categoria (manual para las 48 categorías)
bigram_categoria_map = [
    ('crema dental', 26),
    ('papas fritas', 27),
    ('cepillo dental', 28),
    ('limpiador liq', 29),
    ('color perm', 30),
    ('jab liq', 31),
    ('alimento perros', 32),
    ('jugo polvo', 33),
    ('alimento gatos', 34),
    ('celular libre', 35),
    ('jabon tocador', 36),
    ('limpiador liquido', 37),
    ('jabon liq', 31),  # duplicado con 31
    ('agua gas', 39),
    ('enjuague bucal', 40),
    ('mate cocido', 41),
    ('aceitunas verdes', 42),
    ('sal fina', 43),
    ('dulce leche', 44),
    ('agua mineral', 45),
    ('crema corporal', 46),
    ('jabon liquido', 47),
    ('rollo cocina', 48),
    ('dulce batata', 49),
    ('papel higienico', 50),
    ('trapo piso', 51),
    ('toalla fem', 52),
    ('tableta chocolate', 53),
    ('pan rallado', 54),
    ('huevo chocolate', 55),
    ('plato playo', 56),
    ('crema facial', 57),
    ('barra cereal', 58),
    ('tapa empanada', 59),
    ('crema peinar', 60),
    ('huevo pascua', 61),
    ('leche polvo', 62),
    ('chocolate blanco', 63),
    ('toallas fem', 64),
    ('aloe vera', 65),
    ('maquina afeitar', 66),
    ('toallitas humedas', 67),
    ('snack perros', 68),
    ('alfajor chocolate', 69),
    ('agua sab', 70),
    ('lampara led', 71),
    ('carpeta escolar', 72),
    ('plato postre', 73),
]

print("\n🔍 Asignando productos a categorías...\n")

asignaciones = []

for bigram, id_categoria in bigram_categoria_map:
    # Buscar productos que contienen el bigrama
    productos_match = productos_sin_cat[
        productos_sin_cat['producto'].str.lower().str.contains(bigram, regex=False)
    ]
    
    for _, row in productos_match.iterrows():
        asignaciones.append({
            'id_producto': row['id_producto'],
            'id_categoria': id_categoria,
            'metodo': 'regla',
            'confianza': 0.85,
            'bigram': bigram
        })

print(f"📊 Total asignaciones generadas: {len(asignaciones):,}")

# Eliminar duplicados (un producto puede matchear varios bigramas)
import pandas as pd
df_asignaciones = pd.DataFrame(asignaciones)

# Mantener solo una asignación por producto (la primera)
df_asignaciones_unicas = df_asignaciones.drop_duplicates(subset=['id_producto'], keep='first')

print(f"✅ Asignaciones únicas: {len(df_asignaciones_unicas):,} productos")
print(f"\n📄 Productos clasificados por categoría:")
for id_cat, count in df_asignaciones_unicas['id_categoria'].value_counts().head(15).items():
    print(f"   Categoría {id_cat}: {count:,} productos")

In [0]:
# Crear tabla temporal con las asignaciones
from pyspark.sql.functions import lit, current_timestamp

df_spark = spark.createDataFrame(df_asignaciones_unicas)

# Agregar columnas requeridas
df_insert = df_spark.select(
    'id_producto',
    'id_categoria',
    lit(None).alias('id_subcategoria'),
    'metodo',
    'confianza',
    current_timestamp().alias('fecha_asignacion'),
    lit(None).alias('usuario_asignacion'),
    lit('Clasificado por análisis de bigramas').alias('notas')
)

print("\n📥 Actualizando asignaciones en gold_productos_categorias...\n")

# Crear vista temporal
df_insert.createOrReplaceTempView('temp_asignaciones')

# MERGE: actualizar productos que tienen id_categoria=NULL
result_merge = spark.sql("""
    MERGE INTO workspace.superapp.gold_productos_categorias AS target
    USING temp_asignaciones AS source
    ON target.id_producto = source.id_producto
    WHEN MATCHED AND target.id_categoria IS NULL THEN
        UPDATE SET 
            id_categoria = source.id_categoria,
            id_subcategoria = source.id_subcategoria,
            metodo = source.metodo,
            confianza = source.confianza,
            fecha_asignacion = source.fecha_asignacion,
            usuario_asignacion = source.usuario_asignacion,
            notas = source.notas
""")

print("✅ Actualización completada!\n")

# Verificar resultados
print("📊 Distribución de productos clasificados:\n")

result = spark.sql("""
    SELECT 
        CASE 
            WHEN id_categoria IS NULL THEN 'Sin clasificar'
            ELSE 'Clasificado'
        END as estado,
        COUNT(*) as total_productos
    FROM workspace.superapp.gold_productos_categorias
    GROUP BY estado
    ORDER BY estado DESC
""")

display(result)

# 🎯 Resumen Ejecutivo - Categorización Gold Layer SEPA

## 📊 Estado Final

**Productos Clasificados:**
- **Antes:** 20,615 productos (23.3%)
- **Después:** 28,190 productos (31.8%)
- **Ganancia:** +7,575 productos (+8.5%)

**Productos Pendientes:** 60,474 (68.2%)

---

## 🛠️ Metodología Aplicada

### Fase 1: Keyword Rules (Completado)
- **Categorías creadas:** 25 (IDs 1-25)
- **Productos clasificados:** 20,615
- **Método:** Reglas manuales con LIKE patterns
- **Resultado:** Base sólida pero cobertura limitada

### Fase 2: ML Clustering (Pivotado)
- **Approach:** Embeddings + KMeans (401 clusters)
- **Problema:** Embeddings semánticos no son efectivos para nombres de productos retail
- **Resultado:** 35-40% clusters coherentes, insuficiente
- **Decisión:** Pivotar a análisis de n-gramas

### Fase 3: Bigramas + Filtrado de Marcas (ACTUAL)
- **Categorías creadas:** 48 (IDs 26-73)
- **Productos clasificados:** 7,575 adicionales
- **Método:** Extracción de bigramas + filtro de marcas + matching
- **Cobertura:** Top 50 bigramas = 4.1% de productos sin clasificar
- **Precisión:** ~86% bigramas válidos (43/50)

---

## ✅ Logros

1. **73 categorías ultra-específicas** creadas (vs 20-40 genéricas solicitadas originalmente)
2. **Granularidad correcta:** papel_higienico, papas_fritas, crema_dental, etc. (no "limpieza", "snacks")
3. **45 de 48 categorías nuevas** tienen productos asignados (93.75%)
4. **Filtro de marcas robusto** con 70+ marcas filtradas
5. **Infraestructura extensible:** Fácil agregar más bigramas/categorías

---

## 🔴 Desafíos Restantes

1. **Cobertura baja:** 68% productos aún sin clasificar
2. **Long tail:** Productos con nombres únicos o poco frecuentes
3. **Marcas dominan:** Algunos bigramas capturan marcas en lugar de categorías
4. **Trigramas poco efectivos:** Demasiado específicos, baja cobertura

---

## 🚀 Próximos Pasos Recomendados

### Corto Plazo (1-2 semanas)
1. **Expandir bigramas:** Analizar top 100-200 bigramas (actualmente 50)
2. **Refinar filtro de marcas:** Agregar marcas detectadas en posiciones 51-200
3. **Reglas específicas:** Crear reglas manuales para categorías problemáticas (frutas, verduras, carnes)
4. **Validación manual:** Samplear 100 productos de cada nueva categoría y verificar precisión

### Mediano Plazo (3-4 semanas)
1. **LLM Batch Classification:** Usar Databricks Foundation Models para clasificar lotes de 1,000 productos sin categoría
2. **Embeddings custom:** Entrenar embeddings específicos para retail argentino
3. **Categorías jerárquicas:** Crear subcategorías (ej: arroz_blanco, arroz_integral bajo arroz)
4. **Gold Prices implementation:** Implementar tabla gold_prices que une silver_prices con categorización

### Largo Plazo (1-2 meses)
1. **Mantenimiento continuo:** Pipeline mensual para detectar nuevas categorías emergentes
2. **User feedback loop:** Interfaz para que analistas corrijan/validen categorías
3. **Análisis de precios:** Dashboards con evolución de precios por categoría ultra-específica
4. **Alertas de precios:** Notificaciones cuando categorías específicas suben > X%

---

## 📋 Archivos Clave

- `01_create_gold_tables.py`: Estructura gold_categorias + gold_productos_categorias
- `02_populate_initial_categories.py`: 25 categorías keyword rules
- `03_ml_embeddings_clustering.py`: Approach ML (no exitoso)
- `04_detect_coherent_clusters.py`: **Análisis bigramas (ACTUAL)** ✅
- `gold_layer_addition.py`: Código gold_prices (pendiente agregar a pipeline)

---

## 💬 Conclusión

El análisis de bigramas con filtrado de marcas demostró ser más efectivo que el clustering ML para este caso de uso. Se logró crear **73 categorías ultra-específicas** y clasificar **31.8% de productos**. La estrategia de expandir bigramas + reglas manuales para long tail es el camino recomendado para alcanzar 50-70% de cobertura.

In [0]:
print("\n" + "="*100)
print("🤖 FASE 6: LLM CATEGORY DISCOVERY - Databricks Foundation Models")
print("="*100)

# ============================================================================
# 1. ANALIZAR DISTRIBUCIÓN ACTUAL
# ============================================================================
print("\n1️⃣ Analizando distribución actual...\n")

# Total productos clasificados
clasificados = spark.sql("""
    SELECT COUNT(DISTINCT id_producto) as total
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria IS NOT NULL
""").collect()[0]['total']

# Total categorías
total_categorias = spark.sql("""
    SELECT COUNT(*) as total
    FROM workspace.superapp.gold_categorias
""").collect()[0]['total']

# Productos por categoría
stats = spark.sql("""
    SELECT 
        COUNT(DISTINCT id_producto) as productos_por_categoria
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria IS NOT NULL
    GROUP BY id_categoria
""").toPandas()['productos_por_categoria']

avg_productos = stats.mean()
median_productos = stats.median()
min_productos = stats.min()
max_productos = stats.max()

print(f"📊 Estado Actual:")
print(f"   • Productos clasificados: {clasificados:,}")
print(f"   • Categorías totales: {total_categorias}")
print(f"   • Productos/categoría (promedio): {avg_productos:.0f}")
print(f"   • Productos/categoría (mediana): {median_productos:.0f}")
print(f"   • Rango: {min_productos:.0f} - {max_productos:.0f} productos\n")

# Total sin clasificar
sin_clasificar = spark.sql("""
    SELECT COUNT(DISTINCT id_producto) as total
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria IS NULL
""").collect()[0]['total']

print(f"⚠️  Productos sin clasificar: {sin_clasificar:,}")

# Estimar categorías necesarias
categorias_necesarias = int(sin_clasificar / median_productos)

print(f"\n💡 Estimación:")
print(f"   Para mantener distribución similar (~{median_productos:.0f} productos/categoría)")
print(f"   Necesitamos crear ~{categorias_necesarias} categorías nuevas\n")

print("="*100)

In [0]:
from collections import Counter
import re

print("\n2️⃣ Extrayendo bigramas/trigramas de productos sin clasificar...\n")

# Cargar productos sin clasificar
df_sin_clasificar = spark.sql("""
    SELECT DISTINCT p.producto, p.id_producto
    FROM workspace.superapp.silver_prices p
    LEFT JOIN workspace.superapp.gold_productos_categorias pc ON p.id_producto = pc.id_producto
    WHERE pc.id_categoria IS NULL
""").toPandas()

print(f"📄 Productos únicos sin clasificar: {len(df_sin_clasificar):,}\n")

# Filtro de marcas (70+ marcas ya identificadas)
marcas = [
    'carrefour', 'coto', 'jumbo', 'disco', 'dia', 'arcor', 'nestle', 'kraft', 'bimbo', 
    'molinos', 'dos_anclas', 'marolio', 'colgate', 'dove', 'pantene', 'sedal', 'nivea', 
    'rexona', 'gillette', 'oral_b', 'tex', 'home', 'detroit', 'coca_cola', 'pepsi', 
    'quilmes', 'brahma', 'stella_artois', 'ser', 'sancor', 'la_serenisima', 'ilolay', 
    'milkaut', 'tregar', 'dan', 'cindor', 'la_virginia', 'taragui', 'rosamonte', 
    'nescafe', 'dolca', 'terrabusi', 'bagley', 'toddy', 'savora', 'hellmanns', 
    'natura', 'knorr', 'ades', 'ser', 'silver_shadow', 'siempre_libre', 'moto', 
    'saint_gottard', 'san_remo', 'dog_chow', 'top_life', 'pedigree', 'whiskas', 
    'catit', 'excellent', 'vital_can', 'old_prince', 'samsung', 'lg', 'philips', 
    'xiaomi', 'motorola', 'nokia', 'huawei'
]

# Filtro de unidades de medida y patrones de formato
medidas_y_formato = [
    'x grs', 'x gr', 'x g', 'x ml', 'x cc', 'x cm', 'x kg', 'x uni', 'x u', 'x lt', 'x l',
    'bot ml', 'paq grm', 'paq gr', 'bsa gr', 'cja gr', 'ml bot', 'fco gr', 'pot gr',
    'g paq', 'g bsa', 'g cja', 'g pot', 'g fco', 'g doy',
    'ml aer', 'cc bot', 'x x', 's e', 'pet x', 'doy gr',
    'grm', 'grs', 'ml', 'cc', 'kg', 'lt', 'uni', 'bot', 'paq', 'bsa', 'cja', 'fco', 'pot',
    'aer', 'sch', 'doy', 'fwp', 'env'
]

def limpiar_texto(texto):
    """Normalizar texto para análisis"""
    texto = texto.lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    return ' '.join(texto.split())

def extraer_ngramas(texto, n):
    """Extraer n-gramas de un texto"""
    palabras = texto.split()
    return [' '.join(palabras[i:i+n]) for i in range(len(palabras)-n+1)]

def es_marca(ngrama, marcas):
    """Verificar si el n-grama es una marca"""
    palabras = ngrama.split()
    return any(marca.replace('_', ' ') in palabras or marca.replace('_', '') in ''.join(palabras) for marca in marcas)

def es_medida_o_formato(ngrama, patrones):
    """Verificar si el n-grama es una medida o patrón de formato"""
    # Verificar si contiene algún patrón de medida
    for patron in patrones:
        if patron in ngrama:
            return True
    
    # Verificar si es solo números/letras cortas (formato)
    palabras = ngrama.split()
    if len(palabras) == 2:
        # Patrones como "x 500", "250 ml", etc.
        if any(p in ['x', 'de', 'con', 'sin', 'en', 'el', 'la', 'un', 'una'] for p in palabras):
            return True
    
    return False

# Extraer bigramas y trigramas
print("🔍 Extrayendo n-gramas...")

bigramas_counter = Counter()
trigramas_counter = Counter()

for producto in df_sin_clasificar['producto']:
    texto_limpio = limpiar_texto(producto)
    
    # Bigramas
    for bigrama in extraer_ngramas(texto_limpio, 2):
        if not es_marca(bigrama, marcas) and not es_medida_o_formato(bigrama, medidas_y_formato):
            bigramas_counter[bigrama] += 1
    
    # Trigramas
    for trigrama in extraer_ngramas(texto_limpio, 3):
        if not es_marca(trigrama, marcas) and not es_medida_o_formato(trigrama, medidas_y_formato):
            trigramas_counter[trigrama] += 1

print(f"   • Bigramas únicos: {len(bigramas_counter):,}")
print(f"   • Trigramas únicos: {len(trigramas_counter):,}")

# Filtrar n-gramas con al menos MIN_COUNT productos
MIN_COUNT = 50  # Mínimo 50 productos por cluster (aumentado)

bigramas_validos = {k: v for k, v in bigramas_counter.items() if v >= MIN_COUNT}
trigramas_validos = {k: v for k, v in trigramas_counter.items() if v >= MIN_COUNT}

print(f"\n✅ Filtrados (min {MIN_COUNT} productos):")
print(f"   • Bigramas: {len(bigramas_validos):,}")
print(f"   • Trigramas: {len(trigramas_validos):,}")

# Top 30 bigramas (más para compensar filtrado)
print(f"\n📊 Top 30 bigramas más frecuentes:\n")
for i, (ngrama, count) in enumerate(sorted(bigramas_validos.items(), key=lambda x: x[1], reverse=True)[:30], 1):
    print(f"   {i:2d}. {ngrama:<35} {count:>5,} productos")

# Top 20 trigramas
print(f"\n📊 Top 20 trigramas más frecuentes:\n")
for i, (ngrama, count) in enumerate(sorted(trigramas_validos.items(), key=lambda x: x[1], reverse=True)[:20], 1):
    print(f"   {i:2d}. {ngrama:<45} {count:>5,} productos")

# Guardar para siguiente celda
print(f"\n💾 Variables guardadas: bigramas_validos, trigramas_validos, df_sin_clasificar")

In [0]:
import mlflow.deployments
import json
import time

print("\n3️⃣ Inicializando Databricks Foundation Model API...\n")

# Inicializar cliente
client = mlflow.deployments.get_deploy_client("databricks")

# Cargar categorías existentes como ejemplos
categorias_existentes = spark.sql("""
    SELECT id_categoria, nombre, descripcion
    FROM workspace.superapp.gold_categorias
    ORDER BY id_categoria DESC
    LIMIT 30
""").toPandas()

ejemplos_categorias = "\n".join([
    f"  - {row['nombre']}: {row['descripcion'] if row['descripcion'] else 'productos de ' + row['nombre']}"
    for _, row in categorias_existentes.iterrows()
])

print(f"📚 Ejemplos de categorías existentes (últimas 30):\n{ejemplos_categorias[:500]}...\n")

# ============================================================================
# FUNCIÓN: Llamar a LLM para sugerir categoría
# ============================================================================
def sugerir_categoria_llm(ngrama, productos_ejemplo, categorias_existentes_str):
    """
    Llama a Llama 3.3 70B para sugerir nombre de categoría
    
    Args:
        ngrama: El bigrama/trigrama detectado
        productos_ejemplo: Lista de productos que contienen el n-grama
        categorias_existentes_str: String con ejemplos de categorías
    
    Returns:
        dict con {nombre_sugerido, descripcion, confianza}
    """
    
    prompt = f"""Eres un experto en clasificación de productos de supermercado argentino.

Tienes estas categorías existentes como referencia:
{categorias_existentes_str}

Ahora analiza estos productos sin categorizar:
{chr(10).join(['  - ' + p[:80] for p in productos_ejemplo[:10]])}

Todos estos productos contienen el patrón: "{ngrama}"

Tarea:
1. ¿Debería crearse una categoría nueva para estos productos?
2. Si SÍ: Sugiere un nombre siguiendo el estilo existente (snake_case, específico, en español)
3. Si NO: Indica que no es necesario

Responde SOLO con un JSON válido:
{{
  "crear_categoria": true/false,
  "nombre_sugerido": "nombre_categoria" o null,
  "descripcion": "Descripción breve" o null,
  "razon": "Por qué sí o no crear la categoría"
}}

JSON:"""
    
    try:
        response = client.predict(
            endpoint="databricks-meta-llama-3-3-70b-instruct",
            inputs={
                "messages": [
                    {"role": "system", "content": "Eres un asistente que responde SOLO con JSON válido."},
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0.3,
                "max_tokens": 500
            }
        )
        
        # Extraer respuesta
        content = response['choices'][0]['message']['content']
        
        # Intentar parsear JSON
        # Limpiar markdown code blocks si existen
        content = content.strip()
        if content.startswith('```'):
            content = content.split('\n', 1)[1]
        if content.endswith('```'):
            content = content.rsplit('\n', 1)[0]
        
        resultado = json.loads(content)
        return resultado
        
    except Exception as e:
        print(f"   ⚠️  Error llamando a LLM: {e}")
        return {
            "crear_categoria": False,
            "nombre_sugerido": None,
            "descripcion": None,
            "razon": f"Error: {str(e)}"
        }

# ============================================================================
# PROCESAR BIGRAMAS 51-100 (ITERACIÓN 2)
# ============================================================================
print("🔄 ITERACIÓN 2: Procesando bigramas 51-100 con LLM...\n")

top_bigramas = sorted(bigramas_validos.items(), key=lambda x: x[1], reverse=True)[50:100]

sugerencias = []

for i, (ngrama, count) in enumerate(top_bigramas, 51):
    # Obtener productos ejemplo
    productos_con_ngrama = df_sin_clasificar[
        df_sin_clasificar['producto'].str.lower().str.contains(ngrama, regex=False)
    ]['producto'].head(15).tolist()
    
    print(f"{i:2d}. Analizando '{ngrama}' ({count} productos)...", end=' ')
    
    # Llamar a LLM
    resultado = sugerir_categoria_llm(ngrama, productos_con_ngrama, ejemplos_categorias)
    
    if resultado['crear_categoria']:
        print(f"✅ CREAR: {resultado['nombre_sugerido']}")
        sugerencias.append({
            'ngrama': ngrama,
            'count': count,
            'nombre_sugerido': resultado['nombre_sugerido'],
            'descripcion': resultado['descripcion'],
            'razon': resultado['razon'],
            'productos_ejemplo': productos_con_ngrama[:5]
        })
    else:
        print(f"❌ SKIP: {resultado['razon'][:50]}")
    
    # Rate limiting
    time.sleep(0.5)

print(f"\n\n🎯 RESULTADO ITERACIÓN 2: {len(sugerencias)} categorías sugeridas de 50 analizadas\n")
print("="*100)

In [0]:
import pandas as pd

print("\n4️⃣ REVISAR SUGERENCIAS DE CATEGORÍAS\n")
print("="*100)

if len(sugerencias) == 0:
    print("⚠️  No se generaron sugerencias. Ajusta MIN_COUNT o revisa los n-gramas.\n")
else:
    print(f"\n📊 {len(sugerencias)} CATEGORÍAS SUGERIDAS POR EL LLM\n")
    print("="*100)
    
    for i, sug in enumerate(sugerencias, 1):
        print(f"\n{i}. CATEGORÍA SUGERIDA: {sug['nombre_sugerido']}")
        print("-" * 100)
        print(f"   Patrón: '{sug['ngrama']}'")
        print(f"   Productos afectados: {sug['count']:,}")
        print(f"   Descripción: {sug['descripcion']}")
        print(f"   Razón: {sug['razon']}")
        print(f"\n   Ejemplos de productos:")
        for j, prod in enumerate(sug['productos_ejemplo'], 1):
            print(f"      {j}. {prod[:80]}")
    
    print("\n" + "="*100)
    print("💡 PRÓXIMOS PASOS")
    print("="*100)
    print("\n1️⃣ REVISAR: Lee las sugerencias arriba")
    print("2️⃣ FILTRAR: Elimina de 'sugerencias' las que NO quieras crear")
    print("3️⃣ CREAR: Ejecuta la siguiente celda para crear las categorías aprobadas")
    print("4️⃣ ASIGNAR: Asigna productos a las nuevas categorías")
    print("5️⃣ ITERAR: Repite con los siguientes 50 bigramas\n")
    
    # Crear DataFrame para visualización
    df_sugerencias = pd.DataFrame([{
        'nombre': s['nombre_sugerido'],
        'ngrama': s['ngrama'],
        'productos': s['count'],
        'descripcion': s['descripcion'][:60] + '...' if len(s['descripcion']) > 60 else s['descripcion']
    } for s in sugerencias])
    
    print("\n📊 Resumen de sugerencias:\n")
    display(df_sugerencias)
    
    print(f"\n💾 Variable 'sugerencias' disponible con {len(sugerencias)} categorías")

In [0]:
print("\n5️⃣ CREAR CATEGORÍAS APROBADAS\n")
print("="*100)

if len(sugerencias_validas) == 0:
    print("⚠️  No hay sugerencias para crear\n")
else:
    # Obtener próximo ID disponible
    max_id = spark.sql("""
        SELECT MAX(id_categoria) as max_id
        FROM workspace.superapp.gold_categorias
    """).collect()[0]['max_id']
    
    next_id = max_id + 1
    
    print(f"🆔 Próximo ID disponible: {next_id}\n")
    
    # Preparar datos para insertar
    nuevas_categorias = []
    
    for i, sug in enumerate(sugerencias_validas):
        nuevas_categorias.append({
            'id_categoria': next_id + i,
            'nombre': sug['nombre_sugerido'],
            'nivel': '1',  # STRING, no INT
            'parent_id': None,
            'descripcion': sug['descripcion'],
            'fecha_creacion': pd.Timestamp.now()
        })
    
    # Crear DataFrame Spark
    df_nuevas = spark.createDataFrame(pd.DataFrame(nuevas_categorias))
    
    print(f"📄 Insertando {len(nuevas_categorias)} categorías...\n")
    
    for i, cat in enumerate(nuevas_categorias, 1):
        count = sugerencias_validas[i-1]['count']
        print(f"   {i:2d}. ID {cat['id_categoria']}: {cat['nombre']:<35} ({count:>3,} productos)")
    
    # Insertar en tabla
    df_nuevas.write.mode('append').saveAsTable('workspace.superapp.gold_categorias')
    
    print(f"\n✅ {len(nuevas_categorias)} categorías creadas exitosamente!")
    
    # Guardar mapeo ngrama -> id_categoria para siguiente celda
    ngrama_categoria_map = {
        sug['ngrama']: next_id + i 
        for i, sug in enumerate(sugerencias_validas)
    }
    
    print(f"\n💾 Variable 'ngrama_categoria_map' disponible para asignación\n")
    print("="*100)

In [0]:
print("\n6️⃣ ASIGNAR PRODUCTOS A NUEVAS CATEGORÍAS\n")
print("="*100)

if 'ngrama_categoria_map' not in locals():
    print("⚠️  Primero ejecuta la celda anterior para crear las categorías\n")
else:
    asignaciones = []
    
    print(f"🔍 Buscando productos para {len(ngrama_categoria_map)} categorías...\n")
    
    for ngrama, id_categoria in ngrama_categoria_map.items():
        # Buscar productos que contienen el ngrama
        productos_match = df_sin_clasificar[
            df_sin_clasificar['producto'].str.lower().str.contains(ngrama, regex=False)
        ]
        
        for _, row in productos_match.iterrows():
            asignaciones.append({
                'id_producto': row['id_producto'],
                'id_categoria': id_categoria,
                'metodo': 'llm_discovery',
                'confianza': 0.90,
                'ngrama': ngrama
            })
    
    print(f"📊 Total asignaciones generadas: {len(asignaciones):,}")
    
    # Eliminar duplicados (mantener primera asignación)
    df_asignaciones = pd.DataFrame(asignaciones)
    df_asignaciones_unicas = df_asignaciones.drop_duplicates(subset=['id_producto'], keep='first')
    
    print(f"✅ Asignaciones únicas: {len(df_asignaciones_unicas):,} productos\n")
    
    # Distribución por categoría
    print("📊 Productos por categoría:\n")
    for id_cat, count in df_asignaciones_unicas['id_categoria'].value_counts().items():
        ngrama_usado = [k for k, v in ngrama_categoria_map.items() if v == id_cat][0]
        print(f"   Cat {id_cat}: {count:>5,} productos (patrón: '{ngrama_usado}')")
    
    # Crear DataFrame Spark
    df_spark = spark.createDataFrame(df_asignaciones_unicas)
    
    # Agregar columnas requeridas
    from pyspark.sql.functions import lit, current_timestamp
    
    df_insert = df_spark.select(
        'id_producto',
        'id_categoria',
        lit(None).alias('id_subcategoria'),
        'metodo',
        'confianza',
        current_timestamp().alias('fecha_asignacion'),
        lit('llm_discovery').alias('usuario_asignacion'),
        lit('Clasificado por LLM Discovery').alias('notas')
    )
    
    # Crear vista temporal
    df_insert.createOrReplaceTempView('temp_asignaciones_llm')
    
    print(f"\n📥 Actualizando gold_productos_categorias...\n")
    
    # MERGE: actualizar solo productos con id_categoria=NULL
    spark.sql("""
        MERGE INTO workspace.superapp.gold_productos_categorias AS target
        USING temp_asignaciones_llm AS source
        ON target.id_producto = source.id_producto
        WHEN MATCHED AND target.id_categoria IS NULL THEN
            UPDATE SET 
                id_categoria = source.id_categoria,
                id_subcategoria = source.id_subcategoria,
                metodo = source.metodo,
                confianza = source.confianza,
                fecha_asignacion = source.fecha_asignacion,
                usuario_asignacion = source.usuario_asignacion,
                notas = source.notas
    """)
    
    print("✅ Actualización completada!\n")
    
    # Verificar estado final
    result = spark.sql("""
        SELECT 
            CASE 
                WHEN id_categoria IS NULL THEN 'Sin clasificar'
                ELSE 'Clasificado'
            END as estado,
            COUNT(*) as total_productos
        FROM workspace.superapp.gold_productos_categorias
        GROUP BY estado
        ORDER BY estado DESC
    """)
    
    print("📊 Estado final de clasificación:\n")
    display(result)
    
    print("\n" + "="*100)
    print("🔁 PROCESO ITERATIVO")
    print("="*100)
    print("\nPara continuar descubriendo más categorías:")
    print("   1. Vuelve a la celda 2 (extracción de n-gramas)")
    print("   2. Modifica el rango de bigramas (ej: [51:100])")
    print("   3. Ejecuta celdas 3-6 nuevamente")
    print("   4. Repite hasta alcanzar cobertura deseada\n")

In [0]:
print("\n🔧 CORRIGIENDO PRODUCTOS MAL CLASIFICADOS EN LECHE")
print("="*100)

# 1. Mover dulce de leche a categoría 44 (ya existe)
print("\n1️⃣ Moviendo dulce de leche a categoría 44...")

# Crear vista temporal con IDs a actualizar (DISTINCT para evitar duplicados)
spark.sql("""
    CREATE OR REPLACE TEMP VIEW temp_dulce_leche AS
    SELECT DISTINCT pc.id_producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.silver_prices p ON pc.id_producto = p.id_producto
    WHERE pc.id_categoria = 4
    AND (LOWER(p.producto) LIKE '%dulce%leche%' OR LOWER(p.producto) LIKE '%dulce de leche%')
""")

spark.sql("""
    MERGE INTO workspace.superapp.gold_productos_categorias AS target
    USING temp_dulce_leche AS source
    ON target.id_producto = source.id_producto
    WHEN MATCHED THEN UPDATE SET id_categoria = 44
""")

count_dulce = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria = 44
""").collect()[0]['cnt']

print(f"   ✅ Categoría 44 (Dulce de Leche) ahora tiene {count_dulce:,} productos")

# 2. Mover leche en polvo a categoría 62 (ya existe como subcategoría)
print("\n2️⃣ Moviendo leche en polvo a categoría 62...")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW temp_leche_polvo AS
    SELECT DISTINCT pc.id_producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.silver_prices p ON pc.id_producto = p.id_producto
    WHERE pc.id_categoria = 4
    AND LOWER(p.producto) LIKE '%leche%polvo%'
""")

spark.sql("""
    MERGE INTO workspace.superapp.gold_productos_categorias AS target
    USING temp_leche_polvo AS source
    ON target.id_producto = source.id_producto
    WHEN MATCHED THEN UPDATE SET id_categoria = 62
""")

count_polvo = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria = 62
""").collect()[0]['cnt']

print(f"   ✅ Categoría 62 (Leche en Polvo) ahora tiene {count_polvo:,} productos")

# 3. Quitar chocolate/tabletas de LECHE (poner id_categoria = NULL)
print("\n3️⃣ Quitando chocolate/tabletas de categoría LECHE...")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW temp_chocolate_leche AS
    SELECT DISTINCT pc.id_producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.silver_prices p ON pc.id_producto = p.id_producto
    WHERE pc.id_categoria = 4
    AND (LOWER(p.producto) LIKE '%chocolate%' OR LOWER(p.producto) LIKE '%tableta%')
""")

spark.sql("""
    MERGE INTO workspace.superapp.gold_productos_categorias AS target
    USING temp_chocolate_leche AS source
    ON target.id_producto = source.id_producto
    WHEN MATCHED THEN UPDATE SET id_categoria = NULL
""")

print(f"   ✅ Productos de chocolate removidos de LECHE")

# Verificar qué queda en LECHE (id=4)
remaining = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria = 4
""").collect()[0]['cnt']

print(f"\n📊 Productos restantes en LECHE (id=4): {remaining:,}")
print("   Estos deberían ser leches genéricas sin especificar tipo\n")

print("="*100)

In [0]:
print("\n🔧 CORRIGIENDO PRODUCTOS MAL CLASIFICADOS EN LECHE")
print("="*100)

# 1. Mover dulce de leche a categoría 44 (ya existe)
print("\n1️⃣ Moviendo dulce de leche a categoría 44...")

# Crear vista temporal con IDs a actualizar
spark.sql("""
    CREATE OR REPLACE TEMP VIEW temp_dulce_leche AS
    SELECT pc.id_producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.silver_prices p ON pc.id_producto = p.id_producto
    WHERE pc.id_categoria = 4
    AND (LOWER(p.producto) LIKE '%dulce%leche%' OR LOWER(p.producto) LIKE '%dulce de leche%')
""")

spark.sql("""
    MERGE INTO workspace.superapp.gold_productos_categorias AS target
    USING temp_dulce_leche AS source
    ON target.id_producto = source.id_producto
    WHEN MATCHED THEN UPDATE SET id_categoria = 44
""")

count_dulce = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria = 44
""").collect()[0]['cnt']

print(f"   ✅ Categoría 44 (Dulce de Leche) ahora tiene {count_dulce:,} productos")

# 2. Mover leche en polvo a categoría 62 (ya existe como subcategoría)
print("\n2️⃣ Moviendo leche en polvo a categoría 62...")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW temp_leche_polvo AS
    SELECT pc.id_producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.silver_prices p ON pc.id_producto = p.id_producto
    WHERE pc.id_categoria = 4
    AND LOWER(p.producto) LIKE '%leche%polvo%'
""")

spark.sql("""
    MERGE INTO workspace.superapp.gold_productos_categorias AS target
    USING temp_leche_polvo AS source
    ON target.id_producto = source.id_producto
    WHEN MATCHED THEN UPDATE SET id_categoria = 62
""")

count_polvo = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria = 62
""").collect()[0]['cnt']

print(f"   ✅ Categoría 62 (Leche en Polvo) ahora tiene {count_polvo:,} productos")

# 3. Quitar chocolate/tabletas de LECHE (poner id_categoria = NULL)
print("\n3️⃣ Quitando chocolate/tabletas de categoría LECHE...")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW temp_chocolate_leche AS
    SELECT pc.id_producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.silver_prices p ON pc.id_producto = p.id_producto
    WHERE pc.id_categoria = 4
    AND (LOWER(p.producto) LIKE '%chocolate%' OR LOWER(p.producto) LIKE '%tableta%')
""")

spark.sql("""
    MERGE INTO workspace.superapp.gold_productos_categorias AS target
    USING temp_chocolate_leche AS source
    ON target.id_producto = source.id_producto
    WHEN MATCHED THEN UPDATE SET id_categoria = NULL
""")

print(f"   ✅ Productos de chocolate removidos de LECHE")

# Verificar qué queda en LECHE (id=4)
remaining = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM workspace.superapp.gold_productos_categorias
    WHERE id_categoria = 4
""").collect()[0]['cnt']

print(f"\n📊 Productos restantes en LECHE (id=4): {remaining:,}")
print("   Estos deberían ser leches genéricas sin especificar tipo\n")

print("="*100)